# Project 2: The NEU 5335 Drone Racing Challenge!

![Drone Racing Track](https://static.wixstatic.com/media/4bd4ae_57ef3d6e921f476a8cef1cb6fcb62ae7~mv2.jpg/v1/fill/w_2303,h_467,al_c,q_85,enc_auto/4bd4ae_57ef3d6e921f476a8cef1cb6fcb62ae7~mv2.jpg)

<p align="right">
<i> Image credits: <a href="https://www.aerialsports.tv"> Aerial Sports </a> </i>
</p>

"*Faster, faster, until the thrill of speed overcomes the fear of death!*"

Ladies and gentlemen, boys and girls, children of all ages! Welcome to the 2nd Programming project of CS 5335!! In this project, we'll be flying drones from one place to another and racing through hoops, using path planning with drone dynamics!

Have you ever wondered [how drones fly](https://www.youtube.com/watch?v=gsfkGlSajHQ)? How do they hover in the same position in the sky? How do they rotate about their axes and move towards their intended location in the three-dimensional world? We hope you figure all of this as you reach towards the last cells of this notebook!

If you think that Quidditch is the most enticing sport in the world, then you are terribly mistaken. The coolest sport in today's 21st century world is DRONE RACING and if you don't believe this, you should check out this video!

<center>

[![Drone Racing](https://img.youtube.com/vi/bZvNLuC12R0/0.jpg)](https://www.youtube.com/watch?v=7wFEYnRVjc0 "Drone Racing")

</center>

Really cool, right?! But flying with code is even cooler! By the end of the project, you'll also be able to fly just like this!!

## Instructions
- Project PA2 will be released on November 11th, Tuesday and will be due on November 24, Monday at 11:59 AM.
- You will not be able to save your work in the release version of the assignmnet. You **MUST** save a copy to your Drive before you will be able to save any work.
- There are 25 "TODO" items that require you to write code. You will need to complete all of these.
- There are also reflection questions which require a written response. You can find all of these by searching the notebook for "reflection." Your response should go in a text cell directly under each question.
- We may release an extra credit extension to the assignmnet at a later date

### Logistics and Tips
- You need to submit your code and report on gradescope. Please read the submission guidelines at the end of the notebook.
- START EARLY!! This may take some time.
- Press `Shift`+`Enter` to execute the current code cell, instead of clicking the play button with your mouse.
- Press `b` (b for below) to add a code cell below the current one.
- Press `a` (a for above) to add a code cell above the current one.

Hope you enjoy working on this project as much as we did creating it! Let's [learn to fly!](https://www.youtube.com/watch?v=y3FNW6S2XAM) ^_^


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Implementation Notes
This assignment uses [GTSAM](https://gtsam.org/) to represent rotations, translations, etc. GTSAM is primarily a c++ project which offers Python bindings. Most of the available documentation is written for c++, but in general the c++ documentation is applicable to the python bindings.

The drone dynamics used in this assignment are discussed in detail [here](https://www.roboticsbook.org/S72_drone_actions.html). This is not required reading, but it may enhance your understanding.

### Setup
You'll need to run these cells whenever you fire up collab. But don't worry, it shouldn't take much time!

In [ ]:
## installation

## imports
from typing import List, Tuple, Dict
import math

# %pip uninstall numpy
%pip install numpy==1.25
import numpy as np
import pandas as pd
import unittest
import plotly.express as px
import plotly.graph_objects as gos
%pip install gtsam==4.2
import gtsam

rng = np.random.default_rng(12345)

We import `helpers.py` here that'll help us visualize our trees and maps. `gdown` should work fine since it's just one file, but in case you run into any issues, you can also download the `helpers_obstacles.py` file and upload it manually to your notebooks.

In [ ]:
#downloads the helpers file. This can be commented after running once
!pip install --upgrade --no-cache-dir gdown &> /dev/null
!gdown 1rOEki1n8coTVURLBL7Yh2icbpPc-1fB_

import helpers_obstacles as helpers
# the Drone dynamics model is described in detail in: https://www.roboticsbook.org/S72_drone_actions.html
from helpers_obstacles import Drone, axes, axes_figure

# Part 1: Rapidly-exploring Random Trees (RRT)! In 3 Dimensions!



- RRT again? Yes you read that right! But this time, we're taking it up a notch! 😉
- Given the coordinates of our starting location and the destination location, how does a drone fly from the start to the goal? One easy yet effective way is to execute the RRT algorithm for the drone!
- In this section, we shall implement the RRT algorithm in three dimensions (3D), because well, we live in a 3 dimensional world.
- In the previous assignment, each node in our RRT tree was a (x, y) tuple. Now that we're in three dimensions, we require three coordinates (x, y, z) to represent the position of the drone. We shall use `gtsam.Point3` to represent this.

#### The NEU Drone Racing Track!
Let's first quickly load up the racing track. Take a look at those suspended HOOPS!!! In mid air!! We're gonna do RRT and fly through 'em!

In [ ]:
start_race = gtsam.Pose3(r=gtsam.Rot3.Yaw(math.radians(45)), t=gtsam.Point3(1, 3, 8))
helpers.drone_racing_path(helpers.get_hoops(), start_race, [])
helpers.drone_racing_path_with_obstacles(helpers.get_hoops(), start_race, [], obstacles=helpers.get_obstacles_easy())

- We now code up the helper functions that'll help us execute the RRT algorithm. Please read the docstrings carefully for each of them, the arguments to the function, and the return types.
- You can refer to [Section 5.5.6](https://www.roboticsbook.org/S55_diffdrive_planning.html#a-simple-rrt-implementation) in the textbook for some helpful insights, especially for TODO 3.  

In [ ]:
# TODO 1
def generate_random_point(target: gtsam.Point3) -> gtsam.Point3:
  '''
  This function generates a random node in the 3 dimensional configuration space of (10x10x10) and returns it.
  You must ensure that there is atleast a 20% chance with which the target node itself is returned.

  Hints:
   - Use rng for random number generation

  Arguments:
   - target: gtsam.Point3 (the goal itself!)

  Returns:
   - node: gtsam.Point3 - a random configuration of (x, y, z) which should be within the (10x10x10) space
  '''

  node = None

  ######## Student code here ########

  #raise NotImplementedError("generate_random_point is not implemented")
  if rng.random() < 0.2:
    node = target
  else:
    node = gtsam.Point3(rng.uniform(0, 10), rng.uniform(0, 10), rng.uniform(0, 10))

  print(node)

  ######## End student code  ########

  return node

In [ ]:
# TODO 2
def distance_euclidean(point1: gtsam.Point3, point2: gtsam.Point3) -> float:
  '''
  This function computes the euclidean distance between two 3-D points.

  Hints:
   - Use np.linalg.norm to compute the norm

  Arguments:
   - point1: gtsam.Point3
   - point2: gtsam.Point3

  Returns:
   - distance_euclidean: float
  '''

  distance_euclidean = None

  ######## Student code here ########
  #raise NotImplementedError("distance_euclidean is not implemented")
  distance_euclidean = np.linalg.norm(point1 - point2)

  ######## End student code  ########

  return distance_euclidean

- Once we've sampled a random node, we need to find a potential parent for that node. We can do this by finding that node in the tree which is closest to the newly sampled node.
- A naive way of doing this is to loop through all the nodes in the tree, compute the distance and pick the node with the minimum distance.
- However, for this function, you must use numpy vecotrization and parallelize your implementation. A good resource to learn this: [Link](https://medium.com/data-science/vectorization-must-know-technique-to-speed-up-operations-100x-faster-50b6e89ddd45)

In [ ]:
# TODO 3
def find_nearest_node(rrt: List[gtsam.Point3], node: gtsam.Point3):
  '''
  Given the current RRT tree and the newly sampled node, this function returns the node in the tree which is CLOSEST
  to the newly sampled node, as well as the index of that node.

  This can be done naively by looping through each node and computing the distance, but you need to parralelize it!

  Hints:
   - Refer to the textbook on how to vectorize it

  Arguments:
   - rrt: List[gtsam.Point3] (the current tree)
   - node: gtsam.Point3 (the newly sampled point)

  Returns:
   - nearest_node: gtsam.Point3 (the nearest node!)
   - index: int (the index of the nearest node)
  '''

  nearest_node = None
  index = None

  ######## Student code here ########

  #raise NotImplementedError("find_nearest_node is not implemented")
  distances = np.linalg.norm(np.array(rrt) - node, axis=1)
  index = np.argmin(distances)
  nearest_node = rrt[index]
  ######## End student code  ########

  return nearest_node, index

- Once we've sampled a node and found the node closest to it in the RRT tree, we need to find a node which would take us **in the direction** of the sampled node.
- In this 3-dimensional scenario, we shall call this the `steer_node` as we wish to steer our drone in the direction of the newly sampled node.
- We adopt a simple and naive strategy: we turn towards the target, and drive some **fraction of the distance**. (Hint: refer to the textbook!)

In [ ]:
# TODO 4
def steer_naive(parent: gtsam.Point3, target: gtsam.Point3, fraction = 0.2):
  '''
  Thus function steers the drone towards the target point, going a fraction of the displacement
  It returns the new 'steer_node' which takes us closer to the destination.

  Arguments:
   - parent: gtsam.Point3
   - target: gtsam.Point3

  Returns:
   - steer_node: gtsam.Point3
  '''

  steer_node = None

  ######## Student code here ########

#  raise NotImplementedError("steer_naive is not implemented")
  steer_node = parent + fraction * (target - parent)

  ######## End student code  ########

  return steer_node

In [ ]:
class TestRRT(unittest.TestCase):
  def test_generate_random_point(self):
    for _ in range(5):
      node = generate_random_point(gtsam.Point3(4,5,6))
      assert 0 <= node[0] <= 10
      assert 0 <= node[1] <= 10
      assert 0 <= node[2] <= 10

  def test_distance_euclidean(self):
    pt1 = gtsam.Point3(2.70109492, 4.55796488, 2.93292049)
    pt2 = gtsam.Point3(4, 7, 2)
    self.assertAlmostEqual(distance_euclidean(pt1, pt2), 2.9190804346571446, 2)

  def test_find_nearest_node(self):
    pt1 = gtsam.Point3(1,2,3)
    pt2 = gtsam.Point3(0.90320894, 3.55218386, 3.71979848)
    pt3 = gtsam.Point3(1.52256715, 4.24174709, 3.37583879)
    pt4 = gtsam.Point3(1.56803165, 4.10257537, 2.795647)
    pt5 = gtsam.Point3(2.68087164, 3.63713802, 4.25464017)
    new_point = gtsam.Point3(3.74935314, 3.2575652 , 5.20840562)
    rrt = [pt1, pt2, pt3, pt4, pt5]
    answer, index = find_nearest_node(rrt, new_point)
    assert (answer==pt5).all()

  def test_steer_naive(self):
    pt1 = gtsam.Point3(3.80319106, 2.49123788, 2.60348781)
    pt2 = gtsam.Point3(3.81712339, 0.33173367, 0.51835128)
    answer = gtsam.Point3(3.80597753, 2.05933704, 2.1864605)
    steer_node = steer_naive(pt1, pt2)
    assert(np.allclose(answer, steer_node, atol=1e-2))

suite = unittest.TestSuite()
suite.addTest(TestRRT('test_generate_random_point'))
suite.addTest(TestRRT('test_distance_euclidean'))
suite.addTest(TestRRT('test_find_nearest_node'))
suite.addTest(TestRRT('test_steer_naive'))

unittest.TextTestRunner().run(suite)

### Putting it all together!

Let's now use all the functions coded up above and write the RRT loop. Here's the outline of the algorithm for your reference:

1. Start with the RRT tree containing the start node. Our aim is to grow this tree with every iteration of the loop.
2. Sample out a random node in the configuration space. Make sure you return the target node with a 20% probability.
3. Find the node nearest to the newly sampled node in the current tree, and make sure you keep track of the parent node. For every node that we add to the tree, you also need to store the index of its parent.
3. Find the "steer node" - a node in the direction of the sampled node, and add it to the RRT tree. No need to check for obstacles, we don't have any! 😃
4. Repeat steps 2 and 3 until the distance of the latest node in the tree and the target node is less than the threshold. As soon as this terminating condition is reached, you can return the tree and the parents list.

This function is the generalized RRT function which we shall be utilizing multiple times in this project. **It uses other helper functions as arguments!! So make sure you use the exact names as given in the function header.** This is **not** limited to the functions we've coded above.



In [ ]:
# TODO 5
def run_rrt(start, target, generate_random_node, steer, distance, find_nearest_node, threshold):
  '''
  This function is the main RRT loop and executes the entire RRT algorithm.
  Follow the steps outlined above. You should keep sampling nodes until the terminating condition is met.

  Please use the same function names as given in the function definition.

  Arguments:
   - start: the start node, it could be gtsam.Point3 or gtsam.Pose3.
   - target: the destination node, it could be gtsam.Point3 or gtsam.Pose3.
   - generate_random_node: this function helps us randomly sample a node
   - steer: this function finds the steer node, which takes us closer to our destination
   - distance: this function computes the distance between the two nodes in the tree
   - find_nearest_node: this function finds the nearest node to the randomly sampled node in the tree
   - threshold: float, this is used for the terminating the algorithm

  Returns:
   - rrt: List[gtsam.Point3] or List[gtsam.Pose3], contains the entire tree
   - parents: List[int], contains the index of the parent for each node in the tree
  '''

  rrt = []
  parents = []
  max_iterations = 2000
  rrt.append(start)
  parents.append(-1)

  for i in range(max_iterations):
    ######## Student code here ########

    #raise NotImplementedError("run_rrt is not implemented")
    # Step 1- Generate Random Node
    random_node = generate_random_node(target)
    # Step 2- Find the nearest node on the tree
    nearest_node, index = find_nearest_node(rrt, random_node)
    # Step 3- Steer towards the new node acc. to the stepsixe
    steer_node = steer(nearest_node, random_node)
    # Step 4- Add the new_node point to the tree and index to the parent
    if distance(steer_node, target) < threshold:
      rrt.append(target)
      parents.append(index)
      break
    else:
      rrt.append(steer_node)
      parents.append(index)

    ######## End student code  ########

  return rrt, parents

### Visualizing the RRT Tree and Path!
Alright, with this, we've completed an implementation of the RRT algorithm in 3 dimensions! Let's see what our tree and path look like!

In [ ]:
start_rrt_3d = gtsam.Point3(1,2,3)
target_rrt_3d = gtsam.Point3(4,7,2)
rrt_3d, parents_rrt_3d = run_rrt(start_rrt_3d, target_rrt_3d, generate_random_point, steer_naive, distance_euclidean, find_nearest_node, threshold=0.1)
print("Nodes in RRT: ", len(rrt_3d))

In [ ]:
helpers.visualize_tree(rrt_3d, parents_rrt_3d, start_rrt_3d, target_rrt_3d)

- Remember we kept track of the parents of every node added to the RRT tree? We shall use this information to print out the path from the start node to the target node.
- You need to backtrack and store the node's parent in a `path` list. We've already implemented this function for you 😉

In [ ]:
def get_rrt_path(rrt: List[gtsam.Pose3], parents: List[int]) -> List[gtsam.Pose3]:
  path = []
  i = len(rrt) - 1
  path.append(rrt[i])

  while(parents[i] != -1):
    next = rrt[parents[i]]
    path.append(next)
    i = parents[i]

  path.reverse()
  return path

path_rrt_3d = get_rrt_path(rrt_3d, parents_rrt_3d)
print("Length of Path: ", len(path_rrt_3d))
helpers.visualize_path(path_rrt_3d, start_rrt_3d, target_rrt_3d)

### Reflection Questions:

- If you look closely at the tree and the path, you'll notice that as we reach close to the goal (the green node), the nodes are extremely close to one another! We begin to take really small steps and we converge very slowly towards it. Can you explain why this is happening?

### Response:
The reason for the nodes to be extremely close to one another in the end is probably due to the stepsize. We are biasing the new node generation towards the target but we move only a fraction of that distance. Thus in the final stages, the target node is generated as the next node but we only take a small step in accordance to our stepsize towards it. This results in many small steps finally converging to the target node.

# Part 2: Drone Dynamics

Now that we have a taste for path finding in 3D, let's see how a drone actually flies before we start steering it.

The kinematics of quadrotors and multicopters are those of simple rigid 3D bodies. The kinematics equations are most useful for navigation and control when expressed in the navigation frame $\mathcal{N}$, which for MAV applications is almost universally assumed to be non-rotating and aligned with gravity. In this project, we use the **ENU (East - North - Up)** navigation frame. This means that the x-axis points towards the east, the y-axis points towards the north and the z-axis points upwards.

We also define a body frame $\mathcal{B}$ as having its origin at the center of mass of the vehicle. Following convention in aerospace applications, we fix the the x-axis as pointing to the front of the vehicle (not always the direction of travel), the y-axis as pointing to the left, and the z-axis pointing up, the so-called **FLU (Forward - Left - Up)** convention.

We then define, respectively,
 - the vehicle's position $r^{n}\doteq[x,y,z]^{T}$,
 - its linear velocity $v^n=\dot{r^{n}}\doteq[u,v,w]^{T}$,
 - the attitude $R^n_b\doteq[i^b,j^b,k^b]\in SO(3)$, a $3\times 3$ rotation matrix from $\mathcal{B}$ to $\mathcal{N}$

Above the superscript $n$ and $b$ denote quantities expressed in the *navigation* and *body* frame, respectively.

As we're now using the dynamics of the drone, we also need to represent the orientation of the drone as a rotation matrix. As you may have already guessed, we have an elegant structure that meets our needs - `gtsam.Pose3`.

The attitude and position can be represented by `gtsam.Rot3` and `gtsam.Point3` objects respectively. And they can be combined together to represent the *pose* of the drone using `gtsam.Pose3` objects.

For example, if the current position of the drone is at the origin of $\mathcal{N}$ and $\mathcal{B}$ aligns with $\mathcal{N}$, the attitude is an identity matrix and the position is $[0, 0, 0]$. Let's see how we can represent this in code.

In [ ]:
position = gtsam.Point3(0, 0, 0)
attitude = gtsam.Rot3()
pose = gtsam.Pose3(r = attitude, t = position)

print(f"Position: {position}")
print(f"Attitude: {attitude}")
print(f"Pose: {pose}")

We're going to use the following test suite to check your implementation of the different methods in this part. Feel free to add more test cases in the suite for your own testing.

In [ ]:
class TestDroneDynamics(unittest.TestCase):
  def test_compute_attitude_from_ypr(self):
    yaw = math.radians(45)
    pitch = math.radians(30)
    roll = math.radians(60)

    expected_attitude = gtsam.Rot3(
        [0.612372, 0.612372, -0.5],
        [-0.0473672, 0.65974, 0.75],
        [0.789149, -0.435596, 0.433013]
    )
    actual_attitude = compute_attitude_from_ypr(yaw, pitch, roll)

    assert(actual_attitude.equals(expected_attitude, tol=1e-2))

  def test_compute_force(self):
    attitude = gtsam.Rot3(
        [0.612372, 0.612372, -0.5],
        [-0.0473672, 0.65974, 0.75],
        [0.789149, -0.435596, 0.433013]
    )
    thrust = 20.0

    expected_force = gtsam.Point3(15.78, -8.71, 8.66)
    actual_force = compute_force(attitude, thrust)

    assert(np.allclose(actual_force, expected_force, atol=1e-2))

  def test_compute_terminal_velocity(self):
    force = gtsam.Point3(15.78, -8.71, 8.66)

    expected_terminal_velocity = gtsam.Point3(19.27, -14.32, 14.27)
    actual_terminal_velocity = compute_terminal_velocity(force)

    assert(np.allclose(actual_terminal_velocity, expected_terminal_velocity, atol=1e-2))


The attitude of the drone can also be defined using the yaw, pitch and roll angles of the drone with respect to $\mathcal{N}$. Our first task now will be to compute the attitude of the drone, $R^n_b$, given the yaw, pitch and roll angles of the drone.

In [ ]:
# TODO 6
def compute_attitude_from_ypr(yaw: float, pitch: float, roll: float) -> gtsam.Rot3:
  '''
  Uses yaw, pitch and roll angles to compute the attitude of the drone

  Arguments:
   - yaw: float (in radians)
   - pitch: float (in radians)
   - roll: float (in radians)

  Hint: Use help(gtsam.Rot3) to see different constructors

  Returns:
   - attitude: gtsam.Rot3
  '''

  attitude = None

  ######## Student code here ########

  #raise NotImplementedError("compute_attitude_from_ypr is not implemented")
  attitude = gtsam.Rot3.RzRyRx(roll, pitch, yaw)

  ######## End student code  ########

  return attitude

In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestDroneDynamics('test_compute_attitude_from_ypr'))

unittest.TextTestRunner().run(suite)

Let us take the [Skydio 2+](https://www.skydio.com/skydio-2-plus) as a prototypical and relevant example. Its mass, with battery, is $800g$, but let's just say $1kg$ to make the calculations easier. Its dimensions are $229mm \times 274mm$, but let's assume the motors are at $(\pm 0.10m, \pm 0.10m)$, which is not too far off.

## Flying the drone

The four rotors on the drone provide a thrust upwards (in the body frame) to fly the drone. In general, the force $F^b_z$ aligned with the body z-axis will be:
$$F^b_z = \sum_{i=1}^4 f_i
$$
where $f_i$ is the force applied by each rotor.

Of course, when we *tilt* the quadrotor forwards, we will direct some of that thrust towards generating horizontal acceleration. To get a handle on this, we need to calculate the thrust in the *navigation* frame, but this is just a matter of multiplying with the attitude $R^n_b$:

$$F^n = R^n_b \begin{bmatrix}0 \\ 0 \\ F^b_z\end{bmatrix} = \hat{z}^n_bF^b_z$$

Our next task is to compute the force vector $F^n$, given attitude $R^n_b$ and upwards thrust in the body frame, $F^b_z$.

In [ ]:
# TODO 7
def compute_force(attitude: gtsam.Rot3, thrust: float) -> gtsam.Point3:
  '''
  Computes the force vector given attitude and thrust in the body frame

  Arguments:
   - attitude: gtsam.Rot3, nRb for the drone
   - thrust: float, the upwards thrust produced by the 4 rotors

  Returns:
   - force: gtsam.Point3, the resultant force vector
  '''

  force = None

  ######## Student code here ########

  #raise NotImplementedError("compute_force is not implemented")
  force = attitude.rotate(gtsam.Point3(0, 0, thrust))

  ######## End student code  ########

  return force


In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestDroneDynamics('test_compute_force'))

unittest.TextTestRunner().run(suite)

## Drag and Terminal Velocity

Constant forward thrust, as calculated above, does *not* mean that the drone will continue accelerating, because of *drag*. In fact, from the spec-sheet of the Skydio-2 we see that the top speed (in autonomous mode) is $36\ Mph$, which is about $16\ m/s$, and the theoretical top speed is probably more like $20\ m/s$. Drag force increases *quadratically* with velocity:

$$F_{drag} \propto v^2$$

$$F_{drag} = k_dv^2$$

where $k_d$ is the drag coefficient. Using the specs from Skydio-2, we compute this $k_d = 0.0425$.

This drag force has to exactly balance the maximum forward thrust at terminal velocity:

$$F^n = k_dv_{terminal}^2$$

$$v_{terminal}^2 = \frac{F^n}{k_d} \implies v_{terminal} = \sqrt{\frac{F^n}{k_d}}$$

Our task now is to calculate the terminal velocity vector, $v_{terminal}$, given the force vector, $F^n$ and the drag coefficient, $k_d$.

In [ ]:
# TODO 8
def compute_terminal_velocity(force: gtsam.Point3, kd: float = 0.0425) -> gtsam.Point3:
  '''
  Uses the force vector and drag coefficient to compute the terminal velocity of the drone

  Arguments:
   - force: gtsam.Point3, the force vector in the navigation frame
   - kd: float, drag coefficient

  Returns:
   - terminal_velocity: gtsam.Point3, the maximum velocity vector
  '''

  terminal_velocity = None

  # Add small epsilon in computating force for numerical stability (avoid sqrt(0))
  eps = 1e-6

  ######## Student code here ########

  #raise NotImplementedError("compute_terminal_velocity is not implemented")
  # We need to take force values separatly for the numpy to work.
  force_values = [force[0], force[1], force[2]]
  # Save the signs for final calculations, else we convert the negatives to positives -> Changes the direction of force
  signs = np.sign(force_values)
  force_values = np.sqrt(np.square(force_values))   # Helps find the |x| value
  force_values = (force_values + eps) / kd          # Acc to the formula
  force_values = signs * np.sqrt(force_values)      # Applying the signs back

  print(force_values)
  terminal_velocity = gtsam.Point3(force_values[0], force_values[1], force_values[2])

  ######## End student code  ########

  return terminal_velocity


In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestDroneDynamics('test_compute_terminal_velocity'))

unittest.TextTestRunner().run(suite)

# Part 3: Steering with Terminal Velocity

Now that we know how the thrust applied by the rotors flies the drone with a certain velocity, we can use that to explore the RRT tree.

However, since we learned that the drones now have *poses* instead of just positions, we will need to update our random node generating function to return random poses. Our first task, in this section, is to return a random pose node with the same random position component as before, but a new random attitude component using a random yaw, pitch and roll $\in [-60^\circ, 60^\circ]$.



Again, we're going to use the following test suite to check your implementation of the different methods in this part. Feel free to add more test cases in the suite for your own testing.

In [ ]:
class TestSteeringWithTerminalVelocity(unittest.TestCase):
  def test_generate_random_pose(self):
    target_node = gtsam.Pose3(r = gtsam.Rot3.Yaw(math.radians(45)), t = gtsam.Point3(8, 5, 6))
    for _ in range(5):
      random_node = generate_random_pose(target_node)
      assert(np.all(np.greater_equal(random_node.translation(), gtsam.Point3(0, 0, 0))))
      assert(np.all(np.less_equal(random_node.translation(), gtsam.Point3(10, 10, 10))))
      assert(np.all(np.greater_equal(random_node.rotation().ypr(), gtsam.Point3(math.radians(-60), math.radians(-60), math.radians(-60)))))
      assert(np.all(np.less_equal(random_node.rotation().ypr(), gtsam.Point3(math.radians(60), math.radians(60), math.radians(60)))))

  def test_find_nearest_pose(self):
    rrt_tree = [gtsam.Pose3(
                    r=gtsam.Rot3([1, 0, 0],
                                 [0, 1, 0],
                                 [0, 0, 1]),
                    t=gtsam.Point3(1, 2, 3)),
                gtsam.Pose3(
                    r=gtsam.Rot3([0.771517, -0.617213, 0],
                                 [0.0952381, 0.119048, -0.97619],
                                 [0.617213, 0.771517, 0.154303]),
                    t=gtsam.Point3(2.70427, 3.90543, 3.85213)),
                gtsam.Pose3(
                    r=gtsam.Rot3([0.601649, -0.541882, 0.302815],
                                 [-0.301782, -0.62385, -0.516772],
                                 [0.627501, 0.29376, -0.721074]),
                    t=gtsam.Point3(4.42268, 5.08119, 2.01005)),
                gtsam.Pose3(
                    r=gtsam.Rot3([-0.696943, 0.589581, -0.36631],
                                 [-0.664345, -0.416218, 0.594076],
                                 [0.204431, 0.679463, 0.704654]),
                    t=gtsam.Point3(5.40351, 6.86933, 3.83104)),
                gtsam.Pose3(
                    r=gtsam.Rot3([-0.0686996, 0.218721, -0.818805],
                                 [-0.796488, -0.297401, -0.0126152],
                                 [-0.340626, 0.900832, 0.269211]),
                    t=gtsam.Point3(1.43819, 5.96437, 4.97769))]
    new_node = gtsam.Pose3(
                    r=gtsam.Rot3([0.682707, 0.661423, 0.310534],
                                 [-0.626039, 0.748636, -0.218217],
                                 [-0.376811, -0.0454286, 0.925176]),
                    t=gtsam.Point3(5.65333, 5.65964, 1.60624))
    expected_nearest_node = rrt_tree[2]
    expected_index = 2
    actual_nearest_node, actual_index = find_nearest_pose(rrt_tree, new_node)
    assert(actual_nearest_node.equals(expected_nearest_node, tol=1e-1))
    assert(actual_index == expected_index)

  def test_steer_with_terminal_velocity(self):
    current_node = gtsam.Pose3(gtsam.Rot3.Yaw(math.radians(90)), gtsam.Point3(1, 2, 3))
    new_node = gtsam.Pose3(gtsam.Rot3.Pitch(math.radians(45)), gtsam.Point3(8, 5, 6))

    expected_steer_node = gtsam.Pose3(gtsam.Rot3(
        [0.37, -0.86, 0],
        [0.31, 0.13, -0.87],
        [0.86, 0.37, 0.37]
    ), gtsam.Point3(3.00, 3.31, 4.31))
    actual_steer_node = steer_with_terminal_velocity(current_node, new_node)

    assert(actual_steer_node.equals(expected_steer_node, tol=1e-1))

In [ ]:
# TODO 9
def generate_random_pose(target: gtsam.Pose3) -> gtsam.Pose3:
  '''
  This function generates a random node in the pose configuration space (10x10x10) and returns it.
  You must ensure that there is atleast a 20% chance with which the target node itself is returned.
  The attitude can be randomly sampled via yaw, pitch and roll angles between -60 to 60 degrees.

  Hints:
   - Use rng for random number generation
   - Use compute_attitude_from_ypr function

  Arguments:
   - target: gtsam.Pose3, the target pose for the RRT

  Returns:
   - node: gtsam.Pose3, random pose or target pose
  '''

  node = None

  ######## Student code here ########

  #raise NotImplementedError("generate_random_pose is not implemented")
  if rng.random() < 0.2:
    node = target
  else:
    position = gtsam.Point3(rng.uniform(0, 10), rng.uniform(0, 10), rng.uniform(0, 10))

    roll = rng.uniform(math.radians(-60), math.radians(60))
    pitch = rng.uniform(math.radians(-60), math.radians(60))
    yaw = rng.uniform(math.radians(-60), math.radians(60))
    attitude = compute_attitude_from_ypr(yaw, pitch, roll)

    node = gtsam.Pose3(attitude, position)

  ######## End student code  ########

  return node

In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestSteeringWithTerminalVelocity('test_generate_random_pose'))

unittest.TextTestRunner().run(suite)

Since, we are using poses now, we will also need to update the way we have been calculating the distances between two nodes. You do not need to implement this, you can use the distance function we have provided in `helpers` as `distance_between_poses()`. Run `help(helpers.distance_between_poses)` to know how to use this function.

In [ ]:
# help(helpers.distance_between_poses)

We will now implement the function to find the nearest node in the RRT tree again, using the new distance function. This one is harder to vectorize, so we can implement it in the naive way using a `for` loop.

In [ ]:
# TODO 10
def find_nearest_pose(rrt: List[gtsam.Pose3], node: gtsam.Pose3):
  '''
  This function finds the nearest node in the current RRT tree to the newly sampled node.

  Arguments:
   - rrt: List[gtsam.Pose3] (a list of nodes currently in the tree)
   - node: gtsam.Pose3 (the newly sampled node)

  Returns:
   - nearest: gtsam.Pose3 (the node in the tree which is CLOSEST to the newly sampled node)
   - index: int (the index of the closest node, so we can keep track of the parent)
  '''

  nearest = None
  index = None

  ######## Student code here ########

  #raise NotImplementedError("find_nearest_pose is not implemented")
  min_dist = math.inf
  for i in range(len(rrt)):
    current_distance = helpers.distance_between_poses(rrt[i], node)
    if current_distance < min_dist:
      min_dist = current_distance
      nearest = rrt[i]
      index = i

  ######## End student code  ########

  return nearest, index

In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestSteeringWithTerminalVelocity('test_find_nearest_pose'))

unittest.TextTestRunner().run(suite)

Finally, we can rewrite the `steer()` function used in the RRT algorithm. We want to fly the drone with terminal velocity in the direction of the new random node for a specific duration, and return the node we reach.

In this part, we can assume that we can apply the thrust in the direction we want to steer in, and we always apply the maximum thrust of $20N$. You may want to compute the following:
 - direction of travel
 - new attitude of the drone using `get_new_attitude()` method from `helpers`
 - force vector using `compute_force()`
 - terminal velocity using `compute_terminal_velocity()`
 - new position by applying the velocity for the given duration

In [ ]:
# help(helpers.get_new_attitude)

In [ ]:
# TODO 11
def steer_with_terminal_velocity(current: gtsam.Pose3, target: gtsam.Pose3, duration: float = 0.1) -> gtsam.Pose3:
  '''
  We need to find a short steering from the current pose toward the target pose.
  We fly the drone for a small duration in the direction of the target node at
  the terminal velocity with maximum possible thrust of 20N.

  Arguments:
   - current: gtsam.Pose3, the current pose of the drone
   - target: gtsam.Pose3, the target pose of the drone
   - duration: float, the duration of the flight, default: 0.1

  Returns:
   - steer_node: gtsam.Pose3, the new node reached by the drone
  '''

  steer_node = None

  ######## Student code here ########

  #raise NotImplementedError("steer_with_terminal_velocity is not implemented")
  # First the unit direction vector
  direction = target.translation() - current.translation()
  direction = direction / np.linalg.norm(direction)
  # Then new attitude
  new_attitude = helpers.get_new_attitude(current, direction)
  # Next is force
  force = compute_force(new_attitude, 20)
  # Computing terminal velocity
  terminal_velocity = compute_terminal_velocity(force)
  # New position using terminal velocity
  new_position = current.translation() + terminal_velocity * duration
  steer_node = gtsam.Pose3(new_attitude, new_position)

  ######## End student code  ########

  return steer_node

In [ ]:
help(steer_with_terminal_velocity)

In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestSteeringWithTerminalVelocity('test_steer_with_terminal_velocity'))

unittest.TextTestRunner().run(suite)

### Putting it back into RRT

- Now that we have coded the different components using drone dynamics, let us run the RRT algorithm again with the new steer function!

In [ ]:
start_rrt_drone = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(1, 2, 3))
target_rrt_drone = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(5, 7, 4))
rrt_drone, parents_rrt_drone = run_rrt(start_rrt_drone, target_rrt_drone, generate_random_pose, steer_with_terminal_velocity,
                                       helpers.distance_between_poses, find_nearest_pose, threshold=1.5)
print("Number of RRT Nodes: ", len(rrt_drone))

In [ ]:
helpers.visualize_tree(rrt_drone, parents_rrt_drone, start_rrt_drone, target_rrt_drone)

In [ ]:
path_rrt_drone = get_rrt_path(rrt_drone, parents_rrt_drone)
print("Length of Path: ", len(path_rrt_drone))
print("The path obtained by steering with a terminal velocity:")
helpers.animate_drone_path(path_rrt_drone, start_rrt_drone, target_rrt_drone)

### Reflection Questions

- Is this `steer` function we coded up realistic in nature? What do you think may be missing, if so?

### Response:
The steer function is not realistic in nature because of 2 main reasons:
1. We are conidering the turns to be really instantaneous without considering the inertia. It is naturally not possible to just take a 90 degree without loss of velocity or wasting the energy.
2. The gravitational forces are not being considered into the calculations. The present calculation to calculate the velocity upwards doesnt cancel or reduce considering it has to raise the drone weigth with it. So those extra weight-force has to be taken care of.

# Part 4: Steering with more realistic drone dynamics

Two major limitations stand out in our previous approach of steering the drone:
1. We ignored the gravitational force affecting the direction and magnitude of our applied thrust. If you didn't believe this was important, [let Sheldon and Leonard tell you otherwise](https://www.youtube.com/watch?v=EgXOVGjIeyI).
2. By assuming we could apply the thrust in the direction we want to steer in, we allowed arbitrary and instantaneous attitude changes at the time of steering.

We are going to address these concerns in this section.

Here's the test suite for this part. Feel free to add your test cases for further testing.

In [ ]:
class TestRealisticSteer(unittest.TestCase):
  def test_compute_force_with_gravity(self):
    attitude = gtsam.Rot3(
        [0.612372, 0.612372, -0.5],
        [-0.0473672, 0.65974, 0.75],
        [0.789149, -0.435596, 0.433013]
    )
    thrust = 20.0

    expected_force = gtsam.Point3(15.78, -8.71, -1.34)
    actual_force = compute_force_with_gravity(attitude, thrust)

    assert(np.allclose(actual_force, expected_force, atol=1e-2))

  def test_steer(self):
    current_node = gtsam.Pose3(gtsam.Rot3.Yaw(math.radians(90)), gtsam.Point3(1, 2, 3))
    new_node = gtsam.Pose3(gtsam.Rot3.Pitch(math.radians(45)), gtsam.Point3(8, 5, 6))

    expected_steer_node = gtsam.Pose3(gtsam.Rot3(
        [0.17, 0.97, -0.17],
        [-0.96, 0.20, 0.17],
        [ 0.20, 0.14, 0.97]
    ), gtsam.Point3(1.97, 2.81, 4.49))
    actual_steer_node = steer(current_node, new_node)

    assert(actual_steer_node.equals(expected_steer_node, tol=1e-2))


## Hovering the drone

To hover, assuming $g=10\ m/s^2$, the four rotors have to provide a thrust of $10N$ upwards to compensate for gravity, i.e., $2.5N$ per motor. Of course, we need to be able to accelerate upwards, so let's assume each motor can provide up to double that, i.e., $0$ to $5N$. So, while the drone is level, here are some sample accelerations we can deliver (note that the drone has a mass of $1kg$):
 - $f_i=0N$ for $i\in{1..4}$: downwards acceleration at $-10\ m/s^2$
 - $f_i=2.5N$ for $i\in{1..4}$: stable hover $0\ m/s^2$
 - $f_i=5N$ for $i\in{1..4}$: upwards acceleration at $10\ m/s^2$

### Correcting force vector

We will now correct our implementation of `compute_force` to incorporate the effect of the gravitational force. We can assume $g = 10\ m/s^2$.

In [ ]:
# TODO 12
def compute_force_with_gravity(attitude: gtsam.Rot3, thrust: float, mass: float = 1.0) -> gtsam.Point3:
  '''
  Computes the net force vector given attitude and thrust in the body frame
  by adjusting for the downwards weight force.

  Arguments:
   - attitude: gtsam.Rot3, nRb for the drone
   - thrust: float, the upwards thrust produced by the 4 rotors
   - mass: float, the mass of the drone, default: 1.0

  Returns:
   - force: gtsam.Point3, the resultant force vector
  '''

  force = None
  g = 10.0  # m/s^2

  ######## Student code here ########

  #raise NotImplementedError("compute_force_with_gravity is not implemented")
  force = compute_force(attitude, thrust)
  force[2] = force[2] - mass * g          # Because the z-component is only affected by gravity
                                          # so we do not need to cahnge the rest of the components

  ######## End student code  ########

  return force

In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestRealisticSteer('test_compute_force_with_gravity'))

unittest.TextTestRunner().run(suite)

## Reflection questions
Since we now have the correct drone dynamics in place, let's answer a few reflection questions.

Play with the configurations of yaw, pitch, roll and thrust (in the cell below) and answer these questions for each configuration:
 1. In which direction is the drone flying in the navigation frame?
 2. Is the drone flying upwards, downwards or maintaining level flight?
 3. In what direction is the thrust applied in the navigation frame?
 4. What is the speed of the drone?
 5. What direction is the front of the drone facing in the navigation frame?

Feel free to add your own code to the cell below, if you need it.

**Configurations:** All angles are in degrees and thrust is in Newtons.

| Yaw &nbsp; &nbsp; &nbsp;  &nbsp; &nbsp; &nbsp; | Pitch &nbsp; &nbsp; &nbsp;  &nbsp; &nbsp; &nbsp; | Roll &nbsp; &nbsp; &nbsp;  &nbsp; &nbsp; &nbsp;| Thrust &nbsp; &nbsp; &nbsp;  &nbsp; &nbsp; &nbsp;|
|-----|-------|------|--------|
| 90  |  0    |  0   |  10    |
| 45  |  45   |  45  |  20    |
| 30  |  0    |  45  |  15    |
| 10  |  30   |  30  |  0     |


In [ ]:
# List of test cases: (yaw_deg, pitch_deg, roll_deg, thrust)
test_cases = [
    (90, 0, 0, 10),
    (45, 45, 45, 20),
    (30, 0, 45, 15),
    (10, 30, 30, 0)
]

for i, (yaw_deg, pitch_deg, roll_deg, thrust) in enumerate(test_cases, 1):
    # Controllers
    yaw = math.radians(yaw_deg)
    pitch = math.radians(pitch_deg)
    roll = math.radians(roll_deg)

    # Computing drone velocity
    nRb = compute_attitude_from_ypr(yaw, pitch, roll)
    net_force = compute_force_with_gravity(nRb, thrust)
    terminal_velocity = compute_terminal_velocity(net_force)

    print(f"Orientation in navigation frame (nRb):\n {np.round(nRb.matrix(), 2)}")
    print(f"Net force:\n {np.round(net_force, 2)}")
    print(f"Terminal velocity:\n {np.round(terminal_velocity, 2)}\n\n")

    if i < len(test_cases):
        print("+" * 68 + "\n")

### Response:
1. In which direction is the drone flying in the navigation frame?
2. Is the drone flying upwards, downwards or maintaining level flight?
3. In what direction is the thrust applied in the navigation frame?
4. What is the speed of the drone?
5. What direction is the front of the drone facing in the navigation frame?

Answer:
1. The direction of flight can be calculated/observed from the terminal velocity.
2. The level of flight can be deduced by the net force and the direction in which it points.
3. Thrust in Navigation frame has to be calculated using rotation nRb on the obtained thrust values since they are in the body frame.
4. Speed is calculated by finding the magnitude of the terminal velocity -> ||v||.
5. Direction of drone facing can be found observing the first column of the orientation matrix that gives the values [nXb, nYb, nZb]'.

Values:

Case 1: Yaw, pitch, roll, thrust = [90, 0, 0, 10]
1. Direction of Flight: Stationary. Because terminal velocity = 0.
2. Level of FLight: Hovering. Because thrust perfectly balances the gravity and gives 0 net force in z-axis.
3. Thrust in N Frame: [0, 0, 0].
4. Speed: sqrt(0^2 + 0^2 + 0^2) = 0
5. Front of Drone: +Y direction or North according to ENU convention.

Case 2: Yaw, pitch, roll, thrust = [45, 45, 45, 20]
1. Direction of Flight: Northeast direction because of (+)ve X and some (-)ve Y terminal velocity.
2. Level of FLight: Level flight becasue the z-component of force is 0.
3. Thrust in N Frame: nRb @ [0, 0, 20] = [0.85 * 20, -0.15 * 20, 0.5 * 20] = [17, -3, 10]. South-east direction as the z-component is cancelled out with gravitational force.
4. Speed: sqrt((20.04)^2 + (-8.30)^2 + 0^2) = 21.7m/s
5. Front of Drone: North-east direction with pointing a bit downwards according to ENU convention.

Case 3: Yaw, pitch, roll, thrust = [30, 0, 45, 15]
1. Direction of Flight: According to values of terminal velocity, South-East and upwards. ((+)ve X, (-)ve Y, (+)ve Z values)
2. Level of Flight: Upwards because of positive net force in z-axis.
3. Thrust in N Frame: [5.3, -9.19, 0.61] South-east and upwards.
4. Speed: sqrt((11.17)^2 + (-14.7)^2 + (3.78)^2) = 19m/s
5. Front of Drone: North-east according to ENU convention.

Case 4: Yaw, pitch, roll, thrust = [10, 30, 30, 0]
1. Direction of Flight: According to the terminal velocity values, straight downwards that is vertical descent. (It is falling off!)
2. Level of Flight: Downwards (falling) because there is no thrust to counter the weight/gravitational force.
3. Thrust in N Frame: [0, 0, -10] because quadrotor has no thrust, only the gravitational force so pointing straight down.
4. Speed: sqrt(0^2 + 0^2 + (1.34)^2) = 15.34m/s.
5. Front of Drone: [0.85, 0.15, -0.5] North-east with nose down slightly according to ENU convention.

### Correcting drone attitude while steering

In this section, we will implement a more realistic version of the `steer()` function. We cannot just rotate the drone into any direction of travel. We limit the yaw, pitch and roll rotations of the drone to $[-10^\circ, 10^\circ]$ for a more realistic instantaneous change in attitude. We also attempt to find a thrust value that would take us closest to the new random node, for each yaw, pitch and roll.

To make it more simple, we select a yaw from $3$ values: $[-10^∘, 0^∘, 10^∘]$, a pitch from $3$ values: $[-10^∘, 0^∘, 10^∘]$, a roll from $3$ values: $[-10^∘, 0^∘, 10^∘]$ and a thrust from $4$ values: $[5, 10, 15, 20]$. For each of the $108$ combinations, we find the node we can steer to, by finding the new attitude and applying the terminal velocity computed using the updated force function for a specific duration. We return the node that is closest to the target node we are steering towards.

#### Hint for calculating the new attitude:

The yaw, pitch and roll rotations of $[-10^∘, 0^∘, 10^∘]$ are in the body frame. You can use the existing attitude and this rotation in body frame to compute the new attitude.

$$R^n_{b_1} = R^n_{b_0} R^{b_0}_{b_1}$$

In [ ]:
# TODO 13
def steer(current: gtsam.Pose3, target: gtsam.Pose3, duration = 0.1):
  '''
  Steering with limits on rotation change and thrust values
  using force compuations with gravity

  Arguments:
   - current: gtsam.Pose3, the current pose of the drone
   - target: gtsam.Pose3, the target pose of the drone
   - duration: float, the duration of the flight, default: 0.1

  Returns:
   - steer_node: gtsam.Pose3, the new node reached by the drone
  '''

  steer_node = None
  yaw_values = [-10, 0, 10]
  pitch_values = [-10, 0, 10]
  roll_values = [-10, 0, 10]
  thrust_values = [5, 10, 15, 20]

  ######## Student code here ########
  min_distance = math.inf
#  raise NotImplementedError("steer is not implemented")
  for yaw in yaw_values:
    for pitch in pitch_values:
      for roll in roll_values:
        for thrust in thrust_values:
          # New attitude using new yaw, pitc, roll values
          new_rotation = compute_attitude_from_ypr(math.radians(yaw), math.radians(pitch), math.radians(roll))
          new_attitude = current.rotation() * new_rotation
          # New position value according to new-attitude
          force = compute_force_with_gravity(new_attitude, thrust)
          terminal_velocity = compute_terminal_velocity(force)
          new_position = current.translation() + terminal_velocity * duration
          # New node to be compared using the new-attitude and position
          new_current_position = gtsam.Pose3(new_attitude, new_position)
          new_node = gtsam.Pose3(new_attitude, new_current_position.translation())

          # Finding the nearest new-node to the target. The nearest is selected as the new steer-node.
          if helpers.distance_between_poses(new_node, target) < min_distance:
            min_distance = helpers.distance_between_poses(new_node, target)
            steer_node = new_node
          else:
            continue

  ######## End student code  ########

  return steer_node


In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestRealisticSteer('test_steer'))

unittest.TextTestRunner().run(suite)

### RRT with drone dynamics!

Now that we have coded the appropriate drone dynamics, let us plug the `steer()` method back into our RRT function

In [ ]:
start_rrt_drone_realistic = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(1, 2, 3))
target_rrt_drone_realistic = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(8, 5, 6))
rrt_drone_realistic, parents_rrt_drone_realistic = run_rrt(start_rrt_drone_realistic, target_rrt_drone_realistic,
                                                           generate_random_pose, steer, helpers.distance_between_poses,
                                                           find_nearest_pose, threshold=1.5)
print("Nodes in RRT Tree: ", len(rrt_drone_realistic))

Let us see how our new RRT algorithm now performs with the same start and target poses.

In [ ]:
helpers.visualize_tree(rrt_drone_realistic, parents_rrt_drone_realistic, start_rrt_drone_realistic, target_rrt_drone_realistic)

In [ ]:
path_rrt_drone_realistic = get_rrt_path(rrt_drone_realistic, parents_rrt_drone_realistic)
print("Length of Path: ", len(path_rrt_drone_realistic))
helpers.animate_drone_path(path_rrt_drone_realistic, start_rrt_drone_realistic, target_rrt_drone_realistic)

### Reflection Questions

- Do you think the drone can fly faster than our current implementation using these realistic dynamics? If yes, how?

### Response:
The now may not be faster than it was before. On the other hand it might actually be slower. This is because by cutting of the flexibility to take turns from [60, -60] degrees to [10, -10] degrees we are limiting its movement directions. But this was a necessary step since the intantaneous changes in the direction to the new node makes no sense in the realistic situations. On the other hand the steer function has comparitively no effect on the velocity since it is being accounted for using the thrust and so these changes are not gauranteed tomake the quadrotor go faster.

## Part 5: Drone Racing in Free environment!

- Alright folks, now that we've successfully implemented the RRT algorithm whilst incorporating drone dynamics, it is now time to take your drones to the **NEU 5335 EASY Drone Racing Challenge**!! ARE YOU READY?!?! Check out this exhilarating racing video to get your adrenaline rushing!   

<center>

[![Drone Racing](https://img.youtube.com/vi/dDTqHYbpCVw/0.jpg)](https://www.youtube.com/watch?v=dDTqHYbpCVw "Drone Race")

</center>



- The idea is simple; there are 4 hoops in the NEU Racing Track, which are given to you as `targets`. Your objective is to fly through those hoops in the **same order** and reach the final treasure, which is present at the center of the last hoop in the list.
- We know that RRT plans a path from one start point to one destination point. Since we have multiple hoops to fly through, we can consider them as **intermediate goal points**.
- So you can first use the RRT algorithm to plan a path from the start location to the first hoop (an intermediate goal), then use RRT again to plan your path from that point to the next hoop, and so on and so forth!
- As you would've noticed in Part 4, the path generated by RRT gets us really close to the goal, but doesn't get us exactly there. So we've provided you with a magical function `pass_through_the_hoop`, which performs some wizardry and gets your drone through the hoop 😉
- Now let's have some fun racing the drone for the easy environment and later we will build it for a harder one!

You need to perform RRT with a set of intermediate goal points, use `pass_through_the_hoop` to successfully traverse through the hoops, and obtain your final treasure!

When you call RRT with `run_rrt`, make sure you pass the correct functions as arguments! **You can play around with the threshold, but do NOT exceed a  threshold value of 3**


In [ ]:
# TODO 14
def drone_racing_rrt(start: gtsam.Pose3, targets: List[gtsam.Pose3]) -> List[gtsam.Pose3]:
  '''
  Runs RRT multiple times from start to each target in order
  Note: Since the drone can only reach near the target node, we add
  multiple points at the end of each sub-path to pass through the hoop.
  Use `helpers.pass_through_the_hoop(target, path)` after calculating an RRT path
  to each target for appending the relevant points.

  Arguments:
   - start: gtsam.Pose3, initial position of the drone
   - targets: List[gtsam.Pose3], RRT targets

  Returns:
   - drone_path: List[gtsam.Pose3], entire path from start to last hoop
  '''

  drone_path = []

  ######## Student code here ########

  #raise NotImplementedError("drone_racing_rrt is not implemented")
  # Start is the first position
  current_position = start
  # Loop throuh all the targets in the array
  for target in targets:
    # First fing the rrt path and the parent nodes. This is for all the steps when current position is updated as the last target position.
    rrt_tree, parents = run_rrt(current_position, target, generate_random_pose, steer, helpers.distance_between_poses, find_nearest_pose, threshold=2)
    path = get_rrt_path(rrt_tree, parents)
    # This helps adding waypoints too pass through the loop
    helpers.pass_through_the_hoop(target, path)
    if len(drone_path) == 0:
      drone_path.extend(path)
    else:
      # Skip first point of new segment (it's same as last point of previous)
      drone_path.extend(path[1:])

    # Update current position to last point in this segment
    current_position = path[-1]

  ######## End student code  ########

  return drone_path


In [ ]:
start_rrt_drone_race = gtsam.Pose3(r=gtsam.Rot3(), t=gtsam.Point3(1, 3, 8))
targets_rrt_drone_race = helpers.get_targets()
path_rrt_drone_race = drone_racing_rrt(start_rrt_drone_race, targets_rrt_drone_race)

In [ ]:
helpers.drone_racing_path(helpers.get_hoops(), start_rrt_drone_race, path_rrt_drone_race)
print("Length of Drone Racing Path: ", len(path_rrt_drone_race))

## Part 6.1: RRT with Obstacle Avoidance

### Why Obstacles Matter for Trajectory Optimization

So far, we've successfully navigated the drone through hoops in an **obstacle-free environment**. While our RRT paths work, they have two major limitations:

1. **Jerky trajectories:** RRT paths have sharp turns and discontinuous velocities
2. **No obstacle avoidance:** Real racing environments have pillars, walls, and other drones!

In this section, we'll add **obstacle avoidance** to our RRT planner. This serves as a critical foundation for **Part 7 (Trajectory Optimization)**, where we'll discover that:
- RRT can avoid obstacles but produces rough paths
- Optimization can smooth paths but struggles with collision constraints
- **The best solution:** Use RRT for initial collision-free path, then optimize it!

### The Collision Checking Challenge

When planning with obstacles, we must ensure that:
- Every **node** we add to the RRT tree is collision-free
- Every **edge** (path segment) between nodes doesn't intersect obstacles

**Key modification to RRT:** Before adding a new node, check if the line segment from the nearest node to the new node is collision-free.

### Obstacle Types

We'll work with one obstacle primitive (defined in `helpers.py`) which is a `SphereObstacle(center, radius, name)`: Circular obstacles (pillars, balloons)

The `helpers` module provides collision checking utilities:
- `check_segment_collision(p1, p2, obstacles)`: Returns `True` if segment collides
- `check_path_collision(path, obstacles)`: Validates entire path

In [ ]:
# TODO 15: RRT with Obstacle Avoidance
def run_rrt_with_obstacles(start, target, generate_random_node, steer, distance, find_nearest_node, threshold, obstacles):
  rrt = []
  parents = []

  ######## Student code here ########

  #raise NotImplementedError("run_rrt_with_obstacles is not implemented")
  rrt.append(start)
  parents.append(-1)
  for iter in range(5000):
    random_node = generate_random_node(target)
    nearest_node, index = find_nearest_node(rrt, random_node)
    steer_node = steer(nearest_node, random_node)
    if helpers.check_segment_collision(nearest_node.translation(), steer_node.translation(), obstacles):
      continue
    else:
      if distance(steer_node, target) < threshold:
        rrt.append(target)
        parents.append(index)
        break
      else:
        rrt.append(steer_node)
        parents.append(index)

  ######## End student code  ########

  return rrt, parents

Let's test our RRT with obstacles implementation on a simple scenario before tackling the full racing circuit.


In [ ]:
# Test RRT with obstacles on simple scenario
print("Testing RRT with obstacle avoidance...")

# Simple test: navigate around a single sphere obstacle
start_obs = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(2, 2, 5))
target_obs = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(8, 8, 8))

# Create obstacle in the middle
obstacles_simple = [
    helpers.SphereObstacle(center=[5, 5, 6.5], radius=1.5, name="Central Pillar")
]

# Run RRT with obstacles
rrt_obs, parents_obs = run_rrt_with_obstacles(
    start=start_obs,
    target=target_obs,
    generate_random_node=generate_random_pose,
    steer=steer,
    distance=helpers.distance_between_poses,
    find_nearest_node=find_nearest_pose,
    threshold=1.5,
    obstacles=obstacles_simple
)

print(f"RRT with obstacles completed: {len(rrt_obs)} nodes")

# Extract and verify path
path_obs = get_rrt_path(rrt_obs, parents_obs)
print(f"Path length: {len(path_obs)} waypoints")

# Verify no collisions
has_collision, _ = helpers.check_path_collision(path_obs, obstacles_simple)
if not has_collision:
    print(f"RRT is Collision-free!")
else:
    print(f"RRT may have collisions")


In [ ]:
# Visualize RRT with obstacles
helpers.visualize_path_with_obstacles(
    path_obs, start_obs, target_obs, obstacles_simple
)

## 🏁 PART 6.2: Drone Racing with Obstacles — The Real Challenge

Alright, this is where things get *real*.
So far, you’ve been navigating clean airspace - no walls, no distractions. But now? We’re dropping you into a proper racing arena, full of hoops **and** obstacles. Think of it as “Formula 1 for drones”… but with sphere obstacles termed as chaos 😎.

Let’s see if your planner can handle it.

### The Scenario

Here’s what we’re setting up:

* **4 racing hoops** placed in the environment (same as Part 5).
* **Random obstacles** scattered throughout the course.
* **Goal:** Your drone must pass through **every hoop** in order, without colliding with anything.


### The Setup

We’ll feed this into our RRT planner:

* Each hoop acts as a **waypoint goal** — the RRT will plan from one hoop to the next.
* The environment now has **obstacle volumes** that the planner must avoid.
* The RRT node expansion logic is the same, but now every new edge must pass the *collision check* before being added.


### Why This Is Tough (and Fun)

Let’s break down what makes this scenario interesting (and frustrating in the best way):

- **Jagged RRT paths:**
  With obstacles everywhere, RRT has to wiggle around them. The result? Paths that look like spaghetti, not the clean racing line you want.

- **More nodes = slower planning:**
  The planner will need more samples to find valid routes between hoops. Be patient - this is the trade-off between complexity and realism.

- **Optimization becomes essential:**
  This is *the* reason we care about trajectory optimization (discussed in Part 7). RRT can get you there, but only roughly. To make it *flyable* for a real drone, we’ll soon smooth and optimize it into a continuous, dynamically feasible trajectory.


> **Pro tip:**
> Rewatch that racing video from earlier - the adrenaline will make debugging much less painful.




In [ ]:
# TODO 16: Drone Racing with Obstacle Avoidance
def drone_racing_rrt_with_obstacles(start: gtsam.Pose3, targets: List[gtsam.Pose3],
                                   obstacles: List = None) -> List[gtsam.Pose3]:
  '''
  Navigate through racing hoops while avoiding obstacles.

  This is almost identical to your `drone_racing_rrt` from TODO 14, but uses
  `run_rrt_with_obstacles` instead of `run_rrt`.

  **Algorithm:**
  1. For each hoop:
     a. Run RRT with obstacles from current position to hoop
     b. Extract path
     c. Pass through the hoop (adds waypoints)
     d. Update current position
  2. Return complete racing path

  **Hints:**
  - Copy your `drone_racing_rrt` implementation from TODO 14
  - Replace `run_rrt` call with `run_rrt_with_obstacles`
  - Pass `obstacles` parameter to the RRT function
  - Everything else stays the same!

  **Expected Behavior:**
  - Should navigate around obstacles while reaching hoops
  - May take longer than obstacle-free version
  - Path will be more jagged (sets up need for optimization in Part 7)

  Arguments:
   - start: gtsam.Pose3, initial drone pose
   - targets: List[gtsam.Pose3], hoop center targets
   - obstacles: List[helpers.Obstacle], obstacles to avoid

  Returns:
   - drone_path: List[gtsam.Pose3], complete racing path through all hoops
  '''

  drone_path = []

  ######## Student code here ########

  #raise NotImplementedError("drone_racing_rrt_with_obstacles is not implemented")
  # Start is the first position
  current_position = start
  # Loop throuh all the targets in the array
  for target in targets:
    # First fing the rrt path and the parent nodes. This is for all the steps when current position is updated as the last target position.
    rrt_tree, parents = run_rrt_with_obstacles(current_position, target, generate_random_pose, steer, helpers.distance_between_poses, find_nearest_pose, threshold=2, obstacles=obstacles)
    path = get_rrt_path(rrt_tree, parents)
    # This helps adding waypoints too pass through the loop
    helpers.pass_through_the_hoop(target, path)
    if len(drone_path) == 0:
      drone_path.extend(path)
    else:
      # Skip first point of new segment (it's same as last point of previous)
      drone_path.extend(path[1:])

    # Update current position to last point
    current_position = path[-1]

  ######## End student code  ########

  return drone_path

Now let's run the full drone racing with obstacles!

In [ ]:
# Run drone racing with obstacles!
print("Starting drone racing with obstacles...")

start_racing_obs = gtsam.Pose3(r=gtsam.Rot3(), t=gtsam.Point3(1, 3, 8))
targets_racing_obs = helpers.get_targets()

# Get obstacle configuration (easy difficulty)
obstacles_racing = helpers.get_obstacles_easy()
print(f"Racing with {len(obstacles_racing)} obstacles")

# Execute racing
path_racing_obs = drone_racing_rrt_with_obstacles(
    start_racing_obs,
    targets_racing_obs,
    obstacles_racing
)

In [ ]:
# Visualize racing path with obstacles
helpers.drone_racing_path_with_obstacles(
    hoops=helpers.get_hoops(),
    start=start_racing_obs,
    path=path_racing_obs,
    obstacles=obstacles_racing
)

print(f"\nTotal racing path length: {len(path_racing_obs)} waypoints")

# Verify collision-free
has_collision, _ = helpers.check_path_collision(path_racing_obs, obstacles_racing)
if not has_collision:
    print(f"RRT is Collision-free!")
else:
    print(f"RRT may have collisions")

### Reflection Questions
- What do you think can be some ways to optimize the trajectory taken by the drone? Keep in mind that instantaneous rotation of the drone is limited to -10 to 10 degrees in yaw, pitch and roll.

### Response:
One of the ways to improve the trajectory could be by using the 3 point method similar to the one used in 2D motions. In this we take 3 consecutive points to check if they can be connected without the collsion, but here we add another condition where the angle is between the range [-10, 10] degrees. By doing this we can potentially reduce the number of points in the final path.
Another way could be by reducing the jaggy paths with arch-type paths so that the velocity changes comply with the direction changes. By doin this the velocity will not be affected much and the direction can be changed more easily.


## We Need Trajectory Optimization!!

Look at your racing path visualization. You should notice several issues:

### Problems with RRT Paths:
1. **Jagged trajectories:** Sharp turns and discontinuous velocities
2. **Inefficient routing:** Takes longer routes to avoid obstacles conservatively
3. **Non-smooth controls:** Would cause jerky flight on a real drone
4. **Slow planning:** Thousands of nodes needed for complex environments
5. **No guarantee of optimality:** Path may be much longer than necessary

### This is where trajectory optimization comes in!

In Part 7, we'll learn how to:
- Use RRT to find a **collision-free initial path** (what we just did)
- Feed this path as an **initial guess** to an optimizer
- **Smooth the trajectory** while maintaining collision-free property
- **Minimize energy** and control effort
- **Result:** Smooth racing trajectories!

**The key insight:** RRT is great at finding feasible paths through complex spaces,
but optimization is needed to make them practical for real flight.

Think about it: Would you want to fly on a drone that follows the jagged RRT path,
or on one that follows a smooth optimized trajectory?🚁



---


## Part 7: Trajectory Optimization via Direct Transcription

In this final part, we will smooth and optimize the jagged RRT trajectories using **direct transcription** methods.
This converts trajectory optimization into a large nonlinear programming (NLP) problem that can be solved efficiently. Read about it here: [Link](https://underactuated.mit.edu/trajopt.html)

### Direct Transcription Trajectory Optimization

#### Overview

RRT produces collision-free paths, but they are often jerky and inefficient. We'll use
**direct transcription** to smooth these paths while respecting:
- **Dynamics constraints:** physics must be obeyed between time steps
- **Boundary constraints:** start/goal positions and hoop passage
- **Control limits:** realistic thrust and angular rates
- **Cost minimization:** smooth, efficient trajectories

#### Mathematical Formulation

**State space:** x ∈ ℝ⁶ = [px, py, pz, ψ, θ, φ]  (position + yaw/pitch/roll)

**Control space:** u ∈ ℝ⁴ = [Δψ, Δθ, Δφ, T]  (angular changes + thrust)

**Decision variables:** z ∈ ℝ^(10N+6) = [x₀, x₁, ..., x_N, u₀, u₁, ..., u_{N-1}]

**Optimization problem:**
```
minimize: J(z) = J_thrust + J_angular + J_smoothness + J_gimbal
subject to:
  - Dynamics: x_{k+1} = f(x_k, u_k, dt)  (terminal velocity model)
  - Boundary: x_0 = x_start, x_N at goal, x_h through hoops
  - Bounds: |Δψ|, |Δθ|, |Δφ| ≤ 10°,  5 ≤ T ≤ 20
```

See [MATHEMATICAL_FORMULATION.md](https://drive.google.com/file/d/1nN0UJhSS_eD7XNXYUjCz2may4ssHf5Pf/view?usp=drive_link) for complete understanding.

In [ ]:
# Let's make some Test cases for the helper functions we will use to perform Trajectory Optimization!

class TestTrajOptHelpers(unittest.TestCase):
    """Test angle wrapping, packing, and unpacking functions."""

    def test_angle_diff_small(self):
        """Test angle_diff with small differences."""
        # Small positive difference
        result = angle_diff(0.1, 0.0)
        self.assertAlmostEqual(result, 0.1, places=6)

        # Small negative difference
        result = angle_diff(0.0, 0.1)
        self.assertAlmostEqual(result, -0.1, places=6)

    def test_angle_diff_wrapping(self):
        """Test angle_diff with wrapping around ±π."""
        # Wrapping across +π/-π boundary
        result = angle_diff(np.pi - 0.1, -np.pi + 0.1)
        self.assertAlmostEqual(result, 0.2, places=6)

        # Wrapping the other direction
        result = angle_diff(-np.pi + 0.1, np.pi - 0.1)
        self.assertAlmostEqual(result, -0.2, places=6)

    def test_angle_diff_180_degrees(self):
        """Test angle_diff at exactly 180 degrees."""
        # Exactly π apart (ambiguous, but should handle consistently)
        result = angle_diff(0.0, np.pi)
        self.assertTrue(abs(result - np.pi) < 1e-6 or abs(result + np.pi) < 1e-6)

    def test_pack_decision_vars(self):
        """Test packing states and controls into flat vector."""
        N = 2
        states = np.array([[1, 2, 3, 0.1, 0.2, 0.3],
                           [4, 5, 6, 0.4, 0.5, 0.6],
                           [7, 8, 9, 0.7, 0.8, 0.9]])  # (N+1) x 6
        controls = np.array([[0.01, 0.02, 0.03, 10],
                             [0.04, 0.05, 0.06, 12]])  # N x 4

        z = pack_decision_vars(states, controls, N)

        # Check size
        self.assertEqual(z.shape[0], 26)  # 10*2 + 6 = 26

        # Check first state
        self.assertTrue(np.allclose(z[:6], [1, 2, 3, 0.1, 0.2, 0.3]))

        # Check last state
        self.assertTrue(np.allclose(z[12:18], [7, 8, 9, 0.7, 0.8, 0.9]))

        # Check first control
        self.assertTrue(np.allclose(z[18:22], [0.01, 0.02, 0.03, 10]))

        # Check second control
        self.assertTrue(np.allclose(z[22:26], [0.04, 0.05, 0.06, 12]))

    def test_unpack_decision_vars(self):
        """Test unpacking flat vector into states and controls."""
        N = 2
        # Create a known flat vector
        z = np.array([1, 2, 3, 0.1, 0.2, 0.3,  # state 0
                      4, 5, 6, 0.4, 0.5, 0.6,  # state 1
                      7, 8, 9, 0.7, 0.8, 0.9,  # state 2
                      0.01, 0.02, 0.03, 10,    # control 0
                      0.04, 0.05, 0.06, 12])   # control 1

        states, controls = unpack_decision_vars(z, N)

        # Check shapes
        self.assertEqual(states.shape, (3, 6))
        self.assertEqual(controls.shape, (2, 4))

        # Check first state
        self.assertTrue(np.allclose(states[0, :], [1, 2, 3, 0.1, 0.2, 0.3]))

        # Check last state
        self.assertTrue(np.allclose(states[2, :], [7, 8, 9, 0.7, 0.8, 0.9]))

        # Check controls
        self.assertTrue(np.allclose(controls[0, :], [0.01, 0.02, 0.03, 10]))
        self.assertTrue(np.allclose(controls[1, :], [0.04, 0.05, 0.06, 12]))

    def test_pack_unpack_inverse(self):
        """Test that pack and unpack are inverse operations."""
        N = 3
        # Create random states and controls
        states = np.random.randn(N+1, 6)
        controls = np.random.randn(N, 4)

        # Pack then unpack
        z = pack_decision_vars(states, controls, N)
        states_recovered, controls_recovered = unpack_decision_vars(z, N)

        # Should recover original arrays
        self.assertTrue(np.allclose(states, states_recovered))
        self.assertTrue(np.allclose(controls, controls_recovered))



In [ ]:
# TODO 17: Angle Difference Function
def angle_diff(angle_to: float, angle_from: float) -> float:
    """
    Compute shortest angular difference (wrapped to [-π, π]).

    1) Why This Matters
    Angles wrap around at ±π (180°). The difference between 170° and -170° is NOT 340°, but rather 20° (going the short way around the circle).
    This function is CRITICAL for all angle-related costs and constraints in optimization.

    2) Mathematical Background
    For angles α, β ∈ [-π, π]:
        Δ = wrap(α - β)   where wrap(θ) maps θ to [-π, π]

    3) Physical Interpretation
    When controlling drone attitude, we want the shortest rotation path:
    - From yaw=170° to yaw=-170°: rotate 20° (NOT 340°)
    - From pitch=3° to pitch=-3°: rotate 6° (straightforward)

    Arguments:
        angle_to: target angle (radians)
        angle_from: source angle (radians)

    Returns:
        difference: shortest angular distance (radians), wrapped to [-π, π]
    """

    ######## Student code here ########

    #raise NotImplementedError("angle_diff is not implemented")
    diff = angle_to - angle_from
    # Setting the limits
    if diff > np.pi or diff < -np.pi:
      # If positive, forward turn works. So subtractinf from 2 PI
      if diff >= 0:
        diff = (2 * np.pi) - diff
      # If negative, then we need to take the turn in opposite direction. So subtracting 2 PI
      else:
        diff = -diff - (2 * np.pi)      # -(-ve) value gives positive value of diff
    ######## End student code ########
    return diff



In [ ]:
# Run tests for Part 6 helper functions
suite = unittest.TestSuite()
suite.addTest(TestTrajOptHelpers('test_angle_diff_small'))
suite.addTest(TestTrajOptHelpers('test_angle_diff_wrapping'))
suite.addTest(TestTrajOptHelpers('test_angle_diff_180_degrees'))

unittest.TextTestRunner().run(suite)


In [ ]:
# TODO 18: Pack Decision Variables
def pack_decision_vars(states: np.ndarray, controls: np.ndarray, N: int) -> np.ndarray:
    """
    Pack states and controls into flat decision vector for optimization.

    1) Why This Matters:
    scipy.optimize.minimize requires a single flat vector of decision variables.
    We need to convert our structured trajectory (states and controls at each time step)
    into this flat format, and later unpack it back.

    2) Physical Interpretation
    - **States (first part):** All N+1 waypoints (start, intermediates, goal)
    - **Controls (second part):** All N control inputs (applied between waypoints)

    Arguments:
        states: (N+1) x 6 array [px, py, pz, yaw, pitch, roll]
        controls: N x 4 array [dyaw, dpitch, droll, thrust]
        N: number of time steps

    Returns:
        z: flat vector of length 10*N + 6
    """

    ######## Student code here ########

    #raise NotImplementedError("pack_decision_vars is not implemented")
    z = np.zeros(10*N + 6)
    states = states.flatten()
    controls = controls.flatten()

    z = np.concatenate((states, controls))

    ######## End student code ########
    return z



In [ ]:
# TODO 19: Unpack Decision Variables

def unpack_decision_vars(z: np.ndarray, N: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Unpack flat decision vector into structured states and controls arrays.

    1) Why This Matters
    This is the inverse of `pack_decision_vars`. The optimizer works with flat vectors,
    but our cost/constraint functions need structured arrays. This function extracts
    the trajectory information from the flat optimization variable.

    2) Physical Interpretation
    We're converting from optimizer format (flat vector) back to trajectory format
    (time-indexed waypoints and control sequences).

    Arguments:
        z: flat vector of length 10*N + 6
        N: number of time steps

    Returns:
        states: (N+1) x 6 array [px, py, pz, yaw, pitch, roll]
        controls: N x 4 array [dyaw, dpitch, droll, thrust]
    """

    ######## Student code here ########

    #raise NotImplementedError("unpack_decision_vars is not implemented")
    states = np.zeros((N+1, 6))
    controls = np.zeros((N, 4))

    states = z[:(N+1)*6].reshape((N+1, 6))
    controls = z[(N+1)*6:].reshape((N, 4))


    ######## End student code ########

    return states, controls




In [ ]:
# Run tests for Part 6 helper functions
suite = unittest.TestSuite()

suite.addTest(TestTrajOptHelpers('test_pack_decision_vars'))
suite.addTest(TestTrajOptHelpers('test_unpack_decision_vars'))
suite.addTest(TestTrajOptHelpers('test_pack_unpack_inverse'))

unittest.TextTestRunner().run(suite)

## 7.1 Defining the Cost Function

Now that we can pack and unpack our decision variables, we're ready to define the **cost function** that the optimizer will minimize. This is the heart of trajectory optimization!

### What Are We Optimizing?

Remember, trajectory optimization is about finding the "best" trajectory through the hoops. But what does "best" mean? We need to mathematically define what makes one trajectory better than another.

Our total cost function has the form:

$$
J(z) = J_{\text{thrust}}(z) + J_{\text{angular}}(z) + J_{\text{smooth}}(z) + J_{\text{gimbal}}(z)
$$

Where:
- **$J_{\text{thrust}}$**: Penalizes deviation from hover thrust (energy efficiency)
- **$J_{\text{angular}}$**: Penalizes large angular velocities (aggressive maneuvers)
- **$J_{\text{smooth}}$**: Penalizes control changes between time steps (jerk)
- **$J_{\text{gimbal}}$**: Penalizes dangerous pitch angles near ±90° (gimbal lock)

### Why Multiple Cost Terms?

Each term captures a different aspect of "good" flight:

1. **Thrust Deviation ($J_{\text{thrust}}$)**:
   - Encourages flight near hover condition (T ≈ 10)
   - Hovering is most energy-efficient
   - Large thrust deviations means more battery consumption

2. **Angular Velocity ($J_{\text{angular}}$)**:
   - Penalizes rapid rotations (high $\Delta\psi$, $\Delta\theta$, $\Delta\phi$)
   - Aggressive maneuvers stress motors and risk instability
   - Encourages smooth, gentle turns

3. **Control Smoothness ($J_{\text{smooth}}$)**:
   - Penalizes sudden changes in control inputs
   - Example: Going from T=5 to T=20 in one time step is jerky
   - Smooth control changes → comfortable flight

4. **Gimbal Lock ($J_{\text{gimbal}}$)**:
   - Exponentially penalizes pitch near ±90°
   - At pitch = ±90°, yaw and roll become indistinguishable (singularity!)
   - Keeps drone away from dangerous attitude configurations

### How the Optimizer Uses This

The `scipy.optimize.minimize` function will:
1. Start with an initial guess trajectory (start with the rrt traj)
2. Compute $J(z)$ for current trajectory
3. Try small variations to decrease $J$
4. Repeat until it finds a local minimum

The optimizer tries to make $J$ as small as possible while satisfying all constraints (dynamics, boundary, collisions).

### What You'll Implement

In the next series of TODOs, you will implement each cost term individually:

- **TODO 20**: `cost_function_thrust()` - Penalize thrust deviation from hover
- **TODO 21**: `cost_function_angular_velocity()` - Penalize large rotations
- **TODO 22**: `cost_function_smoothness()` - Penalize control jerk
- **TODO 23**: `cost_function_gimbal_lock()` - Penalize dangerous pitch
- **TODO 24**: `cost_function_integrated()` - Combine all terms with weights

After implementing each cost function, you'll run unit tests to verify correctness. The test cases check:
- ✅ Zero cost for ideal conditions (hover, no rotation, smooth controls)
- ✅ Positive cost for deviations from ideal
- ✅ Large penalties for dangerous configurations

### Mathematical Notation Reminder

- $N$ = number of time steps
- $x_k$ = state at time $k$ = $[p_x, p_y, p_z, \psi, \theta, \phi]_k$
- $u_k$ = control at time $k$ = $[\Delta\psi, \Delta\theta, \Delta\phi, T]_k$
- $\psi$ (yaw), $\theta$ (pitch), $\phi$ (roll) in radians
- $T_{\text{hover}} = 10$ (hover thrust to counteract gravity)

Let's start implementing each cost component!

In [ ]:
# Lets Define some Test Cases for the Cost Functions to check if they are working as expected!
class TestCostFunctions(unittest.TestCase):
    """Test all cost functions for trajectory optimization."""

    def test_cost_function_thrust_hover(self):
        """Test thrust cost at hover condition."""
        N = 2
        states = np.zeros((N+1, 6))
        # All thrust at hover (10 N)
        controls = np.array([[0, 0, 0, 10],
                             [0, 0, 0, 10]])

        cost = cost_function_thrust(states, controls, N, weight=0.1)
        # Thrust exactly at hover, so cost should be 0
        self.assertAlmostEqual(cost, 0.0, places=6)

    def test_cost_function_thrust_deviation(self):
        """Test thrust cost with deviation from hover."""
        N = 2
        states = np.zeros((N+1, 6))
        # Thrust deviates by +2 from hover (10 -> 12)
        controls = np.array([[0, 0, 0, 12],
                             [0, 0, 0, 12]])

        cost = cost_function_thrust(states, controls, N, weight=0.1)
        # Cost = 0.1 * (2^2 + 2^2) = 0.1 * 8 = 0.8
        self.assertAlmostEqual(cost, 0.8, places=6)

    def test_cost_function_angular_zero(self):
        """Test angular cost with zero angular velocities."""
        N = 2
        states = np.zeros((N+1, 6))
        controls = np.array([[0, 0, 0, 10],
                             [0, 0, 0, 10]])

        cost = cost_function_angular(states, controls, N, weight=1.0)
        # All angular velocities zero, cost should be 0
        self.assertAlmostEqual(cost, 0.0, places=6)

    def test_cost_function_angular_nonzero(self):
        """Test angular cost with non-zero angular velocities."""
        N = 2
        states = np.zeros((N+1, 6))
        # Angular velocities: [0.1, 0.2, 0.3]
        controls = np.array([[0.1, 0.2, 0.3, 10],
                             [0.1, 0.2, 0.3, 10]])

        cost = cost_function_angular(states, controls, N, weight=1.0)
        # Cost = 1.0 * ((0.1^2 + 0.2^2 + 0.3^2) + (0.1^2 + 0.2^2 + 0.3^2))
        #      = 1.0 * (0.14 + 0.14) = 0.28
        self.assertAlmostEqual(cost, 0.28, places=6)

    def test_cost_function_smoothness_constant(self):
        """Test smoothness cost with constant controls."""
        N = 3
        states = np.zeros((N+1, 6))
        # Constant controls (no jerk)
        controls = np.array([[0.1, 0.1, 0.1, 10],
                             [0.1, 0.1, 0.1, 10],
                             [0.1, 0.1, 0.1, 10]])

        cost = cost_function_smoothness(states, controls, N, weight=5.0)
        # Controls don't change, so smoothness cost should be 0
        self.assertAlmostEqual(cost, 0.0, places=6)

    def test_cost_function_smoothness_varying(self):
        """Test smoothness cost with varying controls."""
        N = 2
        states = np.zeros((N+1, 6))
        # Controls change from k=0 to k=1
        controls = np.array([[0, 0, 0, 10],
                             [0.1, 0.1, 0.1, 12]])

        cost = cost_function_smoothness(states, controls, N, weight=5.0)
        # Difference: [0.1, 0.1, 0.1, 2]
        # Cost = 5.0 * (0.1^2 + 0.1^2 + 0.1^2 + 2^2) = 5.0 * 4.03 = 20.15
        self.assertAlmostEqual(cost, 20.15, places=6)

    def test_cost_function_gimbal_lock_safe(self):
        """Test gimbal lock penalty in safe range."""
        N = 2
        # Pitch within safe range (< 50°)
        states = np.array([[0, 0, 0, 0, 0.5, 0],      # pitch = 0.5 rad (~29°) - OK
                           [0, 0, 0, 0, 0.6, 0],      # pitch = 0.6 rad (~34°) - OK
                           [0, 0, 0, 0, 0.7, 0]])     # pitch = 0.7 rad (~40°) - OK
        controls = np.zeros((N, 4))

        cost = cost_function_gimbal_lock(states, controls, N)
        # All pitches within safe range, cost should be 0
        self.assertAlmostEqual(cost, 0.0, places=6)

    def test_cost_function_gimbal_lock_danger(self):
        """Test gimbal lock penalty approaching singularity."""
        N = 2
        pitch_limit = 50 * np.pi / 180  # ~0.873 rad
        # Pitch exceeds safe range
        states = np.array([[0, 0, 0, 0, 1.0, 0],      # pitch = 1.0 rad (~57°) - DANGER
                           [0, 0, 0, 0, 0.5, 0],      # pitch = 0.5 rad - OK
                           [0, 0, 0, 0, -1.0, 0]])    # pitch = -1.0 rad - DANGER
        controls = np.zeros((N, 4))

        cost = cost_function_gimbal_lock(states, controls, N)
        # Two states exceed limit
        # Excess for pitch=1.0: 1.0 - 0.873 = 0.127
        # Cost per violation: 1000 * 0.127^2 ≈ 16.13
        # Total: 2 * 16.13 ≈ 32.26
        self.assertTrue(cost > 30.0)  # Should have significant penalty

    def test_cost_function_integration(self):
        """Test integrated cost function combines all costs."""
        N = 2
        states = np.array([[0, 0, 0, 0, 0, 0],
                           [1, 1, 1, 0.1, 0.1, 0.1],
                           [2, 2, 2, 0.2, 0.2, 0.2]])
        controls = np.array([[0.1, 0.1, 0.1, 12],
                             [0.1, 0.1, 0.1, 12]])

        z = pack_decision_vars(states, controls, N)
        weights = {'thrust': 0.1, 'angular': 1.0, 'smoothness': 5.0}

        cost = cost_function_tuned(z, N, weights)

        # Should be sum of individual costs
        cost_thrust = cost_function_thrust(states, controls, N, weights['thrust'])
        cost_angular = cost_function_angular(states, controls, N, weights['angular'])
        cost_smooth = cost_function_smoothness(states, controls, N, weights['smoothness'])
        cost_gimbal = cost_function_gimbal_lock(states, controls, N)

        expected_cost = cost_thrust + cost_angular + cost_smooth + cost_gimbal
        self.assertAlmostEqual(cost, expected_cost, places=6)


In [ ]:
# TODO 20: Cost Function - Thrust Deviation
def cost_function_thrust(states: np.ndarray, controls: np.ndarray, N: int,
                         weight: float = 0.1) -> float:
    """
    Compute thrust deviation cost (penalizes deviation from hover thrust).

    Physical Interpretation

    - Thrust = 10 N: hovering (zero cost contribution)
    - Thrust = 15 N: climbing or accelerating (cost = w × 25)
    - Thrust = 5 N: descending (cost = w × 25)

    Higher thrust -> more power consumption -> higher cost.

    ### Implementation Hints


    Arguments:
        states: (N+1) x 6 array (not used for this cost)
        controls: N x 4 array [dyaw, dpitch, droll, thrust]
        N: number of time steps
        weight: cost weight (default 0.1)

    Returns:
        cost: scalar thrust deviation cost
    """
    T_hover = 10.0
    cost = 0.0
    ######## Student code here ########

    #raise NotImplementedError("cost_function_thrust is not implemented")
    for i in range(N):
      cost += weight * (controls[i][3] - T_hover) ** 2


    ######## End student code ########
    return cost

In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestCostFunctions('test_cost_function_thrust_hover'))
suite.addTest(TestCostFunctions('test_cost_function_thrust_deviation'))

unittest.TextTestRunner().run(suite)

In [ ]:
# TODO 21: Cost Function - Angular Velocity
def cost_function_angular(states: np.ndarray, controls: np.ndarray, N: int,
                          weight: float = 1.0) -> float:
    """
    Compute angular velocity cost (penalizes large attitude changes).

    Physical Interpretation
    - Small angular changes (< 5°): minimal cost, smooth rotation
    - Large angular changes (> 20°): high cost, aggressive maneuver
    - Zero angular change: hovering in place (zero cost)
    Minimizing this encourages the drone to maintain stable orientation.

    Hint: Use numpy operations to compute squared norms efficiently.

    Arguments:
        states: (N+1) x 6 array (not used for this cost)
        controls: N x 4 array [dyaw, dpitch, droll, thrust]
        N: number of time steps
        weight: cost weight (default 1.0)

    Returns:
        cost: scalar angular velocity cost
    """
    cost = 0.0

    ######## Student code here ########

    #raise NotImplementedError("cost_function_angular is not implemented")
    for i in range(N):
      cost += weight * (controls[i][0] ** 2 + controls[i][1] ** 2 + controls[i][2] ** 2)
    ######## End student code ########

    return cost


In [ ]:
suite = unittest.TestSuite()

suite.addTest(TestCostFunctions('test_cost_function_angular_zero'))
suite.addTest(TestCostFunctions('test_cost_function_angular_nonzero'))


unittest.TextTestRunner().run(suite)

In [ ]:
# TODO 22: Cost Function - Control Smoothness
def cost_function_smoothness(states: np.ndarray, controls: np.ndarray, N: int,
                             weight: float = 5.0) -> float:
    """
    Compute control smoothness cost (jerk minimization).

    1) Why This Matters
    Jerk is the derivative of acceleration (third derivative of position).
    High jerk causes:
    - Mechanical wear on actuators
    - Uncomfortable flight dynamics - especially important if we had human passengers
    - Tracking errors in control systems
    Smooth control sequences -> smooth trajectories -> efficient flight.

    2) Physical Interpretation
    - Constant controls over time: zero smoothness cost (ideal)
    - Gradually changing controls: low cost (acceptable)
    - Sudden control jumps: high cost (penalized)
    This encourages smooth transitions, not abrupt changes.

    Hint: Use numpy operations to compute differences and squared norms efficiently.

    Arguments:
        states: (N+1) x 6 array (not used for this cost)
        controls: N x 4 array [dyaw, dpitch, droll, thrust]
        N: number of time steps
        weight: cost weight (default 5.0)

    Returns:
        cost: scalar smoothness cost
    """

    if N <= 1:
        return 0.0

    cost = 0.0

    ######## Student code here ########

    #raise NotImplementedError("cost_function_smoothness is not implemented")

    for i in range(N-1):
      # Actual calculation according to the formula
      #cost += weight * ((controls[i+1][0] - controls[i][0]) ** 2 + (controls[i+1][1] - controls[i][1]) ** 2 + (controls[i+1][2] - controls[i][2]) ** 2 + (controls[i+1][3] - controls[i][3]) ** 2)
      # Shorter form to write using the NumPy fucntion
      cost += weight * np.sum((controls[i+1] - controls[i]) ** 2)

    ######## End student code ########
    return cost


In [ ]:
suite = unittest.TestSuite()

suite.addTest(TestCostFunctions('test_cost_function_smoothness_constant'))
suite.addTest(TestCostFunctions('test_cost_function_smoothness_varying'))

unittest.TextTestRunner().run(suite)

In [ ]:
# TODO 23: Cost Function - Gimbal Lock Penalty
def cost_function_gimbal_lock(states: np.ndarray, controls: np.ndarray, N: int) -> float:
    """
    Soft penalty to avoid gimbal lock singularity.

    1) Why This Matters
    Gimbal lock occurs when pitch = ±90° in Euler angles (ZYX convention).
    At this singularity:
    - Yaw and roll become indistinguishable
    - Rotation matrix loses a degree of freedom
    - Numerical instability in euler_from_rotation_matrix
    We add a soft penalty when |pitch| > 50° to keep the optimizer away from this region.

    2) Physical Interpretation
    - Pitch in [-50°, 50°]: normal operation, zero penalty
    - Pitch approaching ±90°: rapidly increasing penalty
    - Prevents optimizer from exploring gimbal lock region
    This is a **soft constraint** (high cost) rather than a hard constraint.

    Arguments:
        states: (N+1) x 6 array [px, py, pz, yaw, pitch, roll]
        controls: N x 4 array (not used for this cost)
        N: number of time steps

    Returns:
        cost: scalar gimbal lock penalty
    """

    cost = 0.0

    ######## Student code here ########

    #raise NotImplementedError("cost_function_gimbal_lock is not implemented")
    limit = np.deg2rad(50)
    for i in range(N+1):
      if abs(states[i][4]) > limit:
        cost += 1000 * (states[i][4] - limit) ** 2

    ######## End student code ########

    return cost


In [ ]:
suite = unittest.TestSuite()

suite.addTest(TestCostFunctions('test_cost_function_gimbal_lock_safe'))
suite.addTest(TestCostFunctions('test_cost_function_gimbal_lock_danger'))

unittest.TextTestRunner().run(suite)

In [ ]:
# Now we can define the integrated cost function that combines all the individual cost components with tunable weights!
def cost_function_tuned(z: np.ndarray, N: int, weights: Dict[str, float]) -> float:
    """
    Integrated cost function combining all cost components.

    1) Why This Matters
    This is the **objective function** that the optimizer minimizes. It combines
    all the cost components (thrust, angular, smoothness, gimbal lock) with tunable
    weights to achieve desired flight characteristics.

    2) Physical Interpretation
    **Weight Tuning Strategy:**
    - **Racing mode** (fast, aggressive):
      - thrust: 0.05 (allow large thrust changes)
      - angular: 0.5 (allow aggressive turns)
      - smoothness: 5.0 (moderate smoothing)
    - **Normal mode** (smooth, efficient):
      - thrust: 0.1 (prefer hover)
      - angular: 1.0 (smooth turns)
      - smoothness: 10.0 (high smoothing)

    Arguments:
        z: decision variables (flat vector of length 10*N + 6)
        N: number of time steps
        weights: dict with keys 'thrust', 'angular', 'smoothness'

    Returns:
        cost: scalar total cost
    """

    states, controls = unpack_decision_vars(z, N)

    cost = 0.0

    # Add thrust cost
    cost += cost_function_thrust(states, controls, N, weights['thrust'])

    # Add angular velocity cost
    cost += cost_function_angular(states, controls, N, weights['angular'])

    # Add smoothness cost
    cost += cost_function_smoothness(states, controls, N, weights['smoothness'])

    # Add gimbal lock penalty
    cost += cost_function_gimbal_lock(states, controls, N)

    return cost

In [ ]:
suite = unittest.TestSuite()

suite.addTest(TestCostFunctions('test_cost_function_integration'))

unittest.TextTestRunner().run(suite)

### 7.2 Defining Constraints: Making Physics Mandatory

We've defined our cost function $J(z)$ - but cost alone is not enough! Without constraints, the optimizer might find "solutions" that:
- Teleport between waypoints (violate physics)
- Start at the wrong location
- Miss the goal position
- Skip hoops entirely

**Constraints make physically impossible trajectories mathematically impossible.**

#### What Are Constraints?

Constraints are **equations or inequalities** that the optimizer MUST satisfy. There are two types:

1. **Equality Constraints**: $c(z) = 0$ (must be exactly satisfied)
   - Examples: "Start at position (1, 3, 8)", "Obey dynamics", "Pass through hoop center"

2. **Inequality Constraints**: $c(z) \leq 0$ (must be non-positive)
   - Examples: "Stay at least 0.5m from obstacles", "Thrust ≤ 20 N"

The optimizer searches for trajectories $z$ that:
- **Minimize** $J(z)$ (cost)
- **While satisfying** all constraints


#### Our Constraint Types

We implement three types of equality constraints:

##### 1. **Dynamics Constraints** (TODO 24) - THE MOST IMPORTANT

These ensure the trajectory follows the **laws of physics**. For each time step $k = 0, \ldots, N-1$:

**Position dynamics (3 equations):**
$$
\mathbf{p}_{k+1} = \mathbf{p}_k + \mathbf{v}_{\text{terminal}}(\psi_k, \theta_k, \phi_k, T_k) \cdot \Delta t
$$

**Attitude dynamics (3 equations):**
$$
\begin{aligned}
\psi_{k+1} &= \psi_k + \Delta\psi_k \\
\theta_{k+1} &= \theta_k + \Delta\theta_k \\
\phi_{k+1} &= \phi_k + \Delta\phi_k
\end{aligned}
$$

**Total:** $6N$ constraints (6 per time step)


##### 2. **Boundary Constraints** (TODO 25)

These fix the **start and goal** of the trajectory:

**Start constraints (6 equations):**
$$
\mathbf{x}_0 = \mathbf{x}_{\text{start}} \quad \Rightarrow \quad \begin{bmatrix} p_x \\ p_y \\ p_z \\ \psi \\ \theta \\ \phi \end{bmatrix}_0 = \begin{bmatrix} 1 \\ 3 \\ 8 \\ 0 \\ 0 \\ 0 \end{bmatrix}
$$

**Goal position constraints (3 equations):**
$$
\mathbf{p}_N = \mathbf{p}_{\text{goal}} \quad \Rightarrow \quad \begin{bmatrix} p_x \\ p_y \\ p_z \end{bmatrix}_N = \begin{bmatrix} x_{\text{final}} \\ y_{\text{final}} \\ z_{\text{final}} \end{bmatrix}
$$

**Hoop waypoint constraints ($3H$ equations, where $H$ = number of hoops):**
For each hoop $h$, at designated knot index $k_h$:
$$
\mathbf{p}_{k_h} = \mathbf{p}_{\text{hoop}_h}
$$

**Total:** $9 + 3H$ constraints (9 boundary + 3 per hoop)

**Why this matters:** These ensure we start at the right place, end at the right place, and pass through each hoop center.

##### 3. **Collision Constraints** (We define it for you)

These ensure the drone maintains a **safety margin** from obstacles:

**For sphere obstacles:**
$$
c_{\text{sphere}}(z) = (r_{\text{obs}} + r_{\text{safety}}) - \|\mathbf{p}_k - \mathbf{c}_{\text{obs}}\|_2 \leq 0
$$

Equivalently: $\|\mathbf{p}_k - \mathbf{c}_{\text{obs}}\|_2 \geq r_{\text{obs}} + r_{\text{safety}}$

**Total:** Depends on number of obstacles (this is an inequality constraint)

---

#### How Constraints Work Mathematically

For equality constraints $c(z) = 0$, we return the **violation**:

$$
\text{violation} = \text{actual\_value} - \text{expected\_value}
$$

The optimizer adjusts $z$ until all violations are (approximately) zero.


#### Constraint Dimensions Summary

For a trajectory with $N$ time steps and $H$ hoops:

| Constraint Type | Count | Dimension | Type |
|----------------|-------|-----------|------|
| Dynamics | $6N$ | $\mathbb{R}^{6N}$ | Equality |
| Boundary (start) | 6 | $\mathbb{R}^6$ | Equality |
| Boundary (goal) | 3 | $\mathbb{R}^3$ | Equality |
| Hoops | $3H$ | $\mathbb{R}^{3H}$ | Equality |
| **Total Equality** | $6N + 9 + 3H$ | - | - |

**Example:** For $N=15$ time steps and $H=4$ hoops:
- Decision variables: $10(15) + 6 = 156$
- Dynamics constraints: $6(15) = 90$
- Boundary constraints: $9$
- Hoop constraints: $3(4) = 12$
- **Total constraints: $90 + 9 + 12 = 111$ equations**

The optimizer must find 156 variables satisfying 111 equations while minimizing cost!


#### What You'll Implement

- **TODO 24**: `dynamics_constraints_robust()` - Enforce physics (terminal velocity model)
- **TODO 25**: `boundary_constraints_with_hoops()` - Fix start, goal, and hoop passages

After implementing, you'll test each constraint function to verify:
- ✅ Zero violations for valid trajectories
- ✅ Non-zero violations for invalid trajectories
- ✅ Correct dimensions (number of constraint equations)



#### The Big Picture: Cost + Constraints

Our complete optimization problem is:

$$
\boxed{
\begin{aligned}
\min_{z \in \mathbb{R}^{10N+6}} \quad & J(z) & & \text{(minimize cost)} \\
\text{subject to:} \quad & \mathbf{c}_{\text{dyn}}(z) = \mathbf{0} & & \text{(obey physics)} \\
& \mathbf{c}_{\text{boundary}}(z) = \mathbf{0} & & \text{(correct start/goal)} \\
& \mathbf{c}_{\text{hoop}}(z) = \mathbf{0} & & \text{(pass through hoops)} \\
& \mathbf{c}_{\text{collision}}(z) \leq \mathbf{0} & & \text{(avoid obstacles)}
\end{aligned}
}
$$

The optimizer (`scipy.optimize.minimize` with SLSQP method) uses gradient information to efficiently search the space of feasible trajectories for the one with lowest cost.

**Let's implement the constraints!**


In [ ]:
# Lets define some Test Cases for the Constraint Functions to check if they are working as expected!
class TestConstraints(unittest.TestCase):
    """Test dynamics and boundary constraint functions."""

    def test_dynamics_constraints_hover(self):
        """Test dynamics constraints for hovering (stationary) trajectory."""
        N = 2
        dt = 0.1

        # Hovering: same position, zero attitude, hover thrust
        states = np.array([[5, 5, 5, 0, 0, 0],
                           [5, 5, 5, 0, 0, 0],
                           [5, 5, 5, 0, 0, 0]])
        controls = np.array([[0, 0, 0, 10],    # Zero angular changes, hover thrust
                             [0, 0, 0, 10]])

        z = pack_decision_vars(states, controls, N)
        violations = dynamics_constraints_robust(z, N, dt)

        # Check shape: should be 6*N = 12 constraints
        self.assertEqual(violations.shape[0], 6 * N)

        # Hovering should nearly satisfy dynamics (small violations due to drag)
        # Position should change slightly due to terminal velocity
        # But violations should be small
        self.assertTrue(np.max(np.abs(violations)) < 1.0)

    def test_dynamics_constraints_forward_flight(self):
        """Test dynamics constraints for forward flight."""
        N = 2
        dt = 0.1

        # Forward flight: moving in +x direction with pitch
        states = np.array([[0, 0, 5, 0, 0.2, 0],  # pitch forward 0.2 rad
                           [1, 0, 5, 0, 0.2, 0],  # moved 1m forward
                           [2, 0, 5, 0, 0.2, 0]]) # moved 2m total
        controls = np.array([[0, 0, 0, 15],       # More thrust for forward flight
                             [0, 0, 0, 15]])

        z = pack_decision_vars(states, controls, N)
        violations = dynamics_constraints_robust(z, N, dt)

        # Check shape
        self.assertEqual(violations.shape[0], 6 * N)

        # Violations won't be zero (we didn't compute exact dynamics)
        # But they should exist (we're testing the function runs)
        self.assertTrue(isinstance(violations, np.ndarray))

    def test_boundary_constraints_start(self):
        """Test boundary constraints enforce start pose."""
        N = 2
        start_pose = gtsam.Pose3(gtsam.Rot3.Ypr(0.1, 0.2, 0.3),
                                 gtsam.Point3(1, 2, 3))
        goal_position = np.array([8, 9, 10])

        # States that match start pose
        states = np.array([[1, 2, 3, 0.1, 0.2, 0.3],  # Matches start
                           [4, 5, 6, 0.1, 0.2, 0.3],
                           [8, 9, 10, 0.1, 0.2, 0.3]])  # Matches goal position
        controls = np.zeros((N, 4))

        z = pack_decision_vars(states, controls, N)
        violations = boundary_constraints_robust(z, N, start_pose, goal_position, [])

        # Check shape: 6 (start) + 3 (goal) + 0 (hoops) = 9
        self.assertEqual(violations.shape[0], 9)

        # First 6 violations (start pose) should be near zero
        self.assertTrue(np.max(np.abs(violations[:6])) < 0.1)

        # Last 3 violations (goal position) should be near zero
        self.assertTrue(np.max(np.abs(violations[6:9])) < 0.1)

    def test_boundary_constraints_with_hoops(self):
        """Test boundary constraints with hoop waypoints."""
        N = 10
        start_pose = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(0, 0, 0))
        goal_position = np.array([10, 0, 0])

        # Create states with 2 hoops at specific locations
        states = np.zeros((N+1, 6))
        states[:, 0] = np.linspace(0, 10, N+1)  # x from 0 to 10
        # Make sure some states pass near hoop positions
        states[3, :] = [3, 5, 5, 0, 0, 0]   # Near hoop 1
        states[7, :] = [7, 8, 8, 0, 0, 0]   # Near hoop 2
        states[N, :3] = goal_position        # Goal position

        controls = np.zeros((N, 4))

        # Define 2 hoops
        hoop1 = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(3, 5, 5))
        hoop2 = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(7, 8, 8))
        hoops = [hoop1, hoop2]

        z = pack_decision_vars(states, controls, N)
        violations = boundary_constraints_robust(z, N, start_pose, goal_position, hoops)

        # Check shape: 6 (start) + 3 (goal) + 3*2 (hoops) = 15
        self.assertEqual(violations.shape[0], 15)

        # All violations should exist (function runs correctly)
        self.assertTrue(isinstance(violations, np.ndarray))

    def test_boundary_constraints_dimensions(self):
        """Test boundary constraints have correct dimensions for various hoop counts."""
        N = 5
        start_pose = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(0, 0, 0))
        goal_position = np.array([5, 5, 5])

        states = np.random.randn(N+1, 6)
        states[0, :] = [0, 0, 0, 0, 0, 0]
        states[N, :3] = goal_position
        controls = np.zeros((N, 4))
        z = pack_decision_vars(states, controls, N)

        # Test with 0 hoops
        violations = boundary_constraints_robust(z, N, start_pose, goal_position, [])
        self.assertEqual(violations.shape[0], 9)  # 6 + 3 + 0

        # Test with 1 hoop
        hoop1 = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(2, 2, 2))
        violations = boundary_constraints_robust(z, N, start_pose, goal_position, [hoop1])
        self.assertEqual(violations.shape[0], 12)  # 6 + 3 + 3

        # Test with 3 hoops
        hoop2 = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(3, 3, 3))
        hoop3 = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(4, 4, 4))
        violations = boundary_constraints_robust(z, N, start_pose, goal_position, [hoop1, hoop2, hoop3])
        self.assertEqual(violations.shape[0], 18)  # 6 + 3 + 9


In [ ]:
# TODO 24: Dynamics Constraints - Terminal Velocity Model
def dynamics_constraints_robust(z: np.ndarray, N: int, dt: float) -> np.ndarray:
    """
    Enforce physics-based dynamics constraints (terminal velocity model).

    Physical Interpretation
        Think of this as "consistency checking":
        - **Position**: "If I apply thrust T with attitude (ψ,θ,φ), do I end up at p_{k+1}?"
        - **Attitude**: "If I apply angular control Δη, do I get attitude η_{k+1}?"

        The optimizer searches for (states, controls) where physics is respected.

    Hint: Use the 'compute_terminal_velocity' function from TODO 8 to get terminal velocity.

    Arguments:
        z: decision variables (flat vector of length 10*N + 6)
        N: number of time steps
        dt: time step duration (seconds, typically 0.1)

    Returns:
        violations: array of length 6N (3 position + 3 attitude per timestep)
    """
    violations = []
    mass = 1.0  # kg
    g = 10.0    # m/s^2 (gravity)

    ######## Student code here ########
    #raise NotImplementedError("dynamics_constraints_robust is not implemented")
    states, controls = unpack_decision_vars(z, N)

    for i in range(N):
      # Current state
      current_position = states[i, :3]
      yaw, pitch, roll = states[i, 3:]

      # Controls
      thrust = controls[i, 3]
      dy, dp, dr = controls[i, :3]

      # Next states
      next_position = states[i+1, :3]
      yaw_next, pitch_next, roll_next = states[i+1, 3:]

      # Computing terminal velocity using force from compute_force_with_gravity
      current_attitude = compute_attitude_from_ypr(yaw, pitch, roll)
      force = compute_force_with_gravity(current_attitude, thrust, mass)
      terminal_velocity = compute_terminal_velocity(force)

      position_predicted = current_position + terminal_velocity * dt
      violations.extend(next_position - position_predicted)       # 3 values of position

      yaw_predicted = yaw + dy
      pitch_predicted = pitch + dp
      roll_predicted = roll + dr

      # 3 values of the angles -> Total 6 position violations for all N time stamps
      violations.append(angle_diff(yaw_next, yaw_predicted))
      violations.append(angle_diff(pitch_next, pitch_predicted))
      violations.append(angle_diff(roll_next, roll_predicted))


    ######## End student code ########

    return np.array(violations)



In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestConstraints('test_dynamics_constraints_hover'))
suite.addTest(TestConstraints('test_dynamics_constraints_forward_flight'))

unittest.TextTestRunner().run(suite)

In [ ]:
# TODO 25: Boundary Constraints - Start, Goal, and Hoops
def boundary_constraints_robust(z: np.ndarray, N: int, start_pose: gtsam.Pose3,
                                goal_position: np.ndarray, hoops: List[gtsam.Pose3]) -> np.ndarray:
    """
    Enforce start, goal, and hoop waypoint constraints.

    Physical Interpretation
    - Start constraint: "The trajectory must begin exactly where the RRT path starts"
    - Goal constraint: "The trajectory must reach the target position"
    - Hoop constraints: "The trajectory must pass through the center of each hoop"
    The optimizer can adjust intermediate waypoints, but these boundary points are fixed.

    Hints:
    - Use 'helpers.euler_from_rotation_matrix_safe' to extract yaw, pitch, roll from rotation matrix.
    - Use 'helpers.find_hoop_indices_robust' to find which trajectory indices correspond to each hoop.
    - Use 'angle_diff' to compute angular differences (e.g., for yaw, pitch, roll).
    - Remember that start_pose is a full Pose3 (position + orientation), while goal_position is just a 3D point.

    Arguments:
        z: decision variables (flat vector)
        N: number of time steps
        start_pose: initial gtsam.Pose3 (position + orientation)
        goal_position: final position as numpy array [x, y, z]
        hoops: list of gtsam.Pose3 representing hoop positions

    Returns:
        violations: array of length (9 + 3*len(hoops))
    """

    violations = []

    ######## Student code here ########

    #raise NotImplementedError("boundary_constraints_robust is not implemented")
    states, controls = unpack_decision_vars(z, N)
    # Current state
    current_position = states[0, :3]
    yaw, pitch, roll = states[0, 3:]

    # Adding the start position violation (x3)
    c_start = current_position - start_pose.translation()
    violations.extend(c_start)

    # Adding the start pose violation (x3)
    start_rotations = start_pose.rotation().matrix()
    start_yaw, start_pitch, start_roll = helpers.euler_from_rotation_matrix_safe(start_rotations)
    violations.append(angle_diff(yaw, start_yaw))
    violations.append(angle_diff(pitch, start_pitch))
    violations.append(angle_diff(roll, start_roll))

    # Adding the goal position violation (x3)
    final_position = states[N, :3]
    c_goal = final_position - goal_position
    violations.extend(c_goal)

    # Hoop Violations (3 x Number of hoops)
    if len(hoops) > 0:
        # Extract positions from states (N+1 x 3)
        positions = states[:, :3]

        # Extract positions from hoops (convert Pose3 to numpy arrays)
        hoop_positions_list = []
        for hoop in hoops:
            hoop_center = hoop.translation()
            hoop_pos = np.array([hoop_center[0], hoop_center[1], hoop_center[2]])
            hoop_positions_list.append(hoop_pos)

        # Find hoop indices
        hoop_indices = helpers.find_hoop_indices_robust(positions, hoop_positions_list, N)

        # For each hoop, add position constraint
        for h, hoop_pos in enumerate(hoop_positions_list):
            k_h = hoop_indices[h]  # Knot index for this hoop

            # Position at this knot
            position_at_hoop = states[k_h, :3]

            # Add violation (3 values)
            violations.extend(position_at_hoop - hoop_pos)

    ######## End student code ########
    return np.array(violations)

In [ ]:
suite = unittest.TestSuite()
suite.addTest(TestConstraints('test_boundary_constraints_start'))
suite.addTest(TestConstraints('test_boundary_constraints_with_hoops'))
suite.addTest(TestConstraints('test_boundary_constraints_dimensions'))

unittest.TextTestRunner().run(suite)

In [ ]:
# TODO 26 (Done by TAs)
# Now we will define the collision constraints function!
# Look into it to learn how collision avoidance can be implemented for spherical objects in trajectory optimization.

def collision_constraints_optimized(z: np.ndarray, N: int, obstacles: List,
                                   subsample: int = 2) -> np.ndarray:
    """
    Enforce collision avoidance with obstacles (inequality constraints).
    This is challenging because:
    - Inequality constraints are harder to satisfy than equality constraints
    - May make optimization slower or fail to converge
    - Requires careful tuning of safety margins

    Physical Interpretation

    - **Safety margin (0.5m)**: Additional clearance beyond obstacle radius
    - **Subsampling**: Check every Nth knot point (e.g., every 2nd) for efficiency
    - **Sphere obstacles**: Simple distance check
    - **Box obstacles**: More complex (check if inside, compute distance to faces)

    Arguments:
        z: decision variables
        N: number of time steps
        obstacles: list of obstacle objects (SphereObstacle or BoxObstacle)
        subsample: check every 'subsample' knots (e.g., 2 = every other)

    Returns:
        violations: array of inequality constraint violations
        **Note:** This returns violations where:
        - Positive = collision detected
        - Negative = safe
        - The optimizer will try to make all violations ≤ 0
    """

    states, controls = unpack_decision_vars(z, N)
    violations = []
    safety_margin = 0.5

    for k in range(0, N + 1, subsample):
        px, py, pz = states[k, 0:3]
        # point = gtsam.Point3(px, py, pz)

        for obs in obstacles:
            if isinstance(obs, helpers.SphereObstacle):
                # Vectorized distance computation
                dist = np.linalg.norm(np.array([px, py, pz]) - np.array(obs.center))
                violations.append(obs.radius + safety_margin - dist)

    return np.array(violations)


Now Lets initialize from RRT Path to intial guess for Optimization. We are writing this for you!!

In [ ]:
# Initialization from RRT Path
def initialize_from_rrt_robust(rrt_path: List[gtsam.Pose3], N: int, dt: float,
                              start_pose: gtsam.Pose3) -> np.ndarray:
    """
    Convert RRT path to initial guess for optimization.

    1) Why This Matters
    A good initial guess is **critical** for nonlinear optimization convergence:
    - Bad initialization → optimizer gets stuck in local minimum
    - Good initialization → fast convergence to global optimum
    - RRT provides feasible (collision-free) initialization

    2) Physical Interpretation
    We're resampling the RRT path (which may have 50-200 waypoints) down to N+1
    knot points (typically 15-30) for optimization. Linear interpolation provides
    smooth transitions.

    Arguments:
        rrt_path: list of gtsam.Pose3 from RRT
        N: number of time steps for optimization
        dt: time step duration (not used but included for future extensions)
        start_pose: initial pose (for validation)

    Returns:
        z_init: initial decision vector of length 10*N + 6
    """

    path_length = len(rrt_path)
    states_init = np.zeros((N + 1, 6))

    # Resample RRT path to N+1 knot points
    for i in range(N + 1):
        # Linear interpolation index
        idx_float = i * (path_length - 1) / N
        idx_low = int(np.floor(idx_float))
        idx_high = min(int(np.ceil(idx_float)), path_length - 1)
        alpha = idx_float - idx_low

        # Interpolate position
        pos_low = rrt_path[idx_low].translation()
        if idx_high > idx_low:
            pos_high = rrt_path[idx_high].translation()
            pos = pos_low + alpha * (pos_high - pos_low)
        else:
            pos = pos_low

        # Interpolate attitude (linear on Euler angles with wrapping)
        R_low = rrt_path[idx_low].rotation().matrix()
        yaw_low, pitch_low, roll_low = helpers.euler_from_rotation_matrix_safe(R_low)

        if idx_high > idx_low:
            R_high = rrt_path[idx_high].rotation().matrix()
            yaw_high, pitch_high, roll_high = helpers.euler_from_rotation_matrix_safe(R_high)

            # Handle angle wrapping
            yaw = yaw_low + alpha * angle_diff(yaw_high, yaw_low)
            pitch = pitch_low + alpha * angle_diff(pitch_high, pitch_low)
            roll = roll_low + alpha * angle_diff(roll_high, roll_low)
        else:
            yaw, pitch, roll = yaw_low, pitch_low, roll_low

        states_init[i, :] = [pos[0], pos[1], pos[2], yaw, pitch, roll]

    # Initialize controls: compute from state differences
    controls_init = np.zeros((N, 4))
    for k in range(N):
        # Angle changes
        controls_init[k, 0] = angle_diff(states_init[k + 1, 3], states_init[k, 3])  # dyaw
        controls_init[k, 1] = angle_diff(states_init[k + 1, 4], states_init[k, 4])  # dpitch
        controls_init[k, 2] = angle_diff(states_init[k + 1, 5], states_init[k, 5])  # droll

        # Thrust: start with hover thrust
        controls_init[k, 3] = 10.0  # Newtons

    # Clamp controls to bounds (vectorized)
    deg_to_rad = np.pi / 180
    controls_init[:, 0:3] = np.clip(controls_init[:, 0:3], -10 * deg_to_rad, 10 * deg_to_rad)
    controls_init[:, 3] = np.clip(controls_init[:, 3], 5, 20)

    return pack_decision_vars(states_init, controls_init, N)


## 7.3 Putting It All Together: The Optimizer! 🚀

Okay, take a deep breath. You've just implemented a LOT of math:
- ✅ Cost functions (thrust, angular, smoothness, gimbal lock)
- ✅ Dynamics constraints (physics!)
- ✅ Boundary constraints (start, goal, hoops)
- ✅ Collision constraints (obstacles, optional)
- ✅ Helper functions (pack, unpack, angle wrapping)

**Now comes the REALLY cool part** - we're going to unleash `scipy.optimize.minimize` to find the optimal trajectory!


### What Does the Optimizer Actually Do?

Remember this beast of an optimization problem?

$$
\boxed{
\begin{aligned}
\min_{z \in \mathbb{R}^{10N+6}} \quad & J(z) & & \text{(your cost functions)} \\
\text{subject to:} \quad & \mathbf{c}_{\text{dyn}}(z) = \mathbf{0} & & \text{(your dynamics constraints)} \\
& \mathbf{c}_{\text{boundary}}(z) = \mathbf{0} & & \text{(your boundary constraints)} \\
& \mathbf{c}_{\text{collision}}(z) \leq \mathbf{0} & & \text{(your collision constraints)}
\end{aligned}
}
$$


The optimizer is going to:
1. **Start** with your RRT path as an initial guess (156 variables for N=15!)
2. **Compute** the cost $J(z)$ and all constraint violations
3. **Use gradients** (calculus!) to figure out which direction to move $z$
4. **Take a step** that decreases cost while respecting constraints
5. **Repeat** steps 2-4 until convergence (typically 20-50 iterations for a free environment but can get complex for a obstacle course)

This is **nonlinear constrained optimization** - one of the most powerful tools in robotics!


### The Magic of SLSQP (Sequential Least Squares Programming)

We use `scipy.optimize.minimize` with the `SLSQP` method. Why SLSQP?

- ✅ **Handles equality AND inequality constraints** (most optimizers can't do both!)
- ✅ **Gradient-based** = fast convergence (beats genetic algorithms by 100x)
- ✅ **Battle-tested** = used in aerospace, robotics

**How it works (simplified):**
1. Linearize cost and constraints around current point
2. Solve a quadratic program (QP) to get search direction
3. Line search to find best step size
4. Update decision variables
5. Check convergence (gradient norm, constraint violations)


### What Makes Optimization Succeed or Fail?

**Success factors:**
- ✅ Good RRT initialization (feasible path)
- ✅ Correct constraint dimensions (you tested these!)
- ✅ Balanced cost weights (not too large, not too small)
- ✅ Reasonable N value (15-30 knots works well)

**Failure modes:**
- ❌ Bad initialization (path goes through obstacle)
- ❌ Constraint dimension mismatch (crashes immediately)
- ❌ Poorly scaled costs (one term dominates)
- ❌ Too many knots (N > 50 becomes slow)
- ❌ Infeasible problem (impossible to reach goal while avoiding obstacles)

**When optimization fails, we have a fallback strategy:**
1. Try SLSQP first (fast, but sensitive)
2. If SLSQP fails, try `trust-constr` (slower, more robust)
3. If both fail, return RRT path (still collision-free!)


### The Complete Optimization Workflow

Here's what happens when you call `optimize_trajectory()`:

```python
optimized_path, success, info = optimize_trajectory(
    rrt_path=rrt_path,        # Your RRT path from Part 5
    start_pose=start_pose,    # Starting pose
    goal_position=goal_pos,   # Goal position
    hoops=hoops,              # List of hoop poses
    obstacles=obstacles,      # List of obstacles
    N=20,                     # Number of knot points
    dt=0.1,                   # Time step
    weights={'thrust': 0.1, 'angular': 1.0, 'smoothness': 5.0}
)
```

**Step-by-step what happens inside:**

1. **Initialize**
   - Resample RRT path to N+1 knot points
   - Compute initial controls
   - Pack into decision vector $z_0$

2. **Define objective function** for scipy
   ```python
   def objective(z):
       states, controls = unpack_decision_vars(z, N)
       cost = cost_function_integrated(states, controls, N, weights)
       return cost
   ```

3. **Define equality constraints** for scipy
   ```python
   eq_constraints = [
       {'type': 'eq', 'fun': lambda z: dynamics_constraints_robust(z, N, dt)},
       {'type': 'eq', 'fun': lambda z: boundary_constraints_robust(z, N, start, goal, hoops)}
   ]
   ```

4. **Define inequality constraints** (if obstacles present)
   ```python
   ineq_constraints = [
       {'type': 'ineq', 'fun': lambda z: -collision_constraints_optimized(z, N, obstacles)}
   ]
   # Note: scipy wants g(z) >= 0, but we return violations where positive = bad
   # So we negate: -collision_constraints makes negative violations become positive (good)
   ```

5. **Call scipy.optimize.minimize**
   ```python
   result = scipy.optimize.minimize(
       objective,
       z_init,
       method='SLSQP',
       constraints=eq_constraints + ineq_constraints,
       options={'maxiter': 400, 'ftol': 1e-5}
   )
   ```

6. **Extract optimized trajectory**
   ```python
   z_opt = result.x
   states_opt, controls_opt = unpack_decision_vars(z_opt, N)
   # Convert states to Pose3 list for visualization
   ```

7. **Validate solution**
   - Check constraint violations < 0.01
   - Check cost is reasonable
   - Check path is collision-free

---

### What You've Built: A Production-Grade Trajectory Optimizer!

Let's put this in perspective. You've implemented the **same core algorithm** used by:
- 🚁 **Drone racing companies** (Skydio, DJI)
- 🚀 **SpaceX** (Falcon 9 landing trajectories)
- 🚗 **Self-driving cars** (motion planning)
- 🤖 **Humanoid robots** (Boston Dynamics Atlas)
- ✈️ **Aircraft autopilots** (Boeing, Airbus)

The only differences are:
- More complex dynamics models (yours is 6-DOF, theirs might be 12-DOF)
- More sophisticated cost functions (fuel optimization, passenger comfort)
- Real-time implementation (MPC: Model Predictive Control)

**But the core math? Identical.** Direct transcription + constrained optimization = industry standard.

---

### Your TODOs Are Done - Now Let's FLY! 🎉

You've completed the TODOs. All the hard math is implemented. Now we get to the fun part:

**In the next cells, you'll:**
- ✅ Run optimization on simple paths (no hoops)
- ✅ Run optimization on racing paths (with hoops!)
- ✅ Compare RRT vs optimized paths side-by-side
- ✅ Visualize velocity and acceleration profiles
- ✅ See your drone smoothly flying through hoops
- ✅ Challenge yourself with obstacles

**The optimizer functions (`optimize_trajectory` and `optimize_racing_path_sequential`) are PROVIDED** because:
- They're mostly boilerplate scipy code
- The real learning was in implementing cost/constraint functions (which you did!)
- You'll learn more by USING them and understanding the results

Think of it like this:
- **You built the engine** (cost functions, constraints)
- **We provided the chassis** (scipy wrapper code)
- **Now let's race!** 🏎️

Ready? Let's optimize some trajectories!

### 7.3.1 Simple Path Optimization (No Hoops)

Let's start simple - optimizing a path from point A to B without hoops.

**What to watch for:**
- RRT path (angular, suboptimal)
- Optimized path (smooth, efficient)
- Cost reduction
- Constraint satisfaction

In [ ]:
# Define simple scenario
start_simple = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(2, 2, 5))
goal_simple = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(8, 8, 5))

print("Step 1: Running RRT...")
rrt_simple, parents_simple = run_rrt(
    start_simple, goal_simple,
    generate_random_pose, steer, helpers.distance_between_poses,
    find_nearest_pose, threshold=2.0
)
path_simple = get_rrt_path(rrt_simple, parents_simple)
print(f"RRT path: {len(path_simple)} waypoints")

print("\nStep 2: Optimizing trajectory...")
optimized_simple, success_simple, info_simple = helpers.optimize_trajectory(
    rrt_path=path_simple,
    start_pose=start_simple,
    goal_position=goal_simple.translation(),
    hoops=[],
    obstacles=[],
    N=20,
    dt=0.1,
    weights={'thrust': 0.1, 'angular': 1.0, 'smoothness': 5.0},
    # Pass student-implemented functions (TODOs 17-28)
    initialize_from_rrt_robust=initialize_from_rrt_robust,
    dynamics_constraints_robust=dynamics_constraints_robust,
    boundary_constraints_robust=boundary_constraints_robust,
    collision_constraints_optimized=collision_constraints_optimized,
    cost_function_tuned=cost_function_tuned,
    unpack_decision_vars=unpack_decision_vars
)

if success_simple:
    print(f"\n✅ SUCCESS! Cost: {info_simple['cost']:.2f}, Iterations: {info_simple['iterations']}")
else:
    print(f"\n❌ Optimization failed, using RRT path")

print("\nStep 3: Visualizing...")

# Visualize comparison
fig = helpers.visualize_rrt_vs_optimized_comparison(
    path_simple, optimized_simple, start_simple, goal_simple,
    title="Demo 1: RRT vs Optimized"
)
fig.show()



### What Just Happened?

- **RRT path (cyan)**: Angular, explores randomly
- **Optimized path (magenta)**: Smooth, nearly straight
- **Cost reduction**: The optimizer minimized thrust, angular velocity, and jerk
- **Physics respected**: All dynamics constraints satisfied

This is trajectory optimization in action! 💪

### 7.3.2 Racing with Hoops (No Obstacles)
Now let's tackle the real challenge - navigating through 4 hoops!
We will first redo the RRT path from drone racing and then optimize this path for comparison!


In [ ]:
# Get racing setup
hoops_demo2 = helpers.get_hoops()
targets_demo2 = helpers.get_targets()
start_demo2 = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(1, 3, 8))

print("Step 1: Running RRT through hoops...")
rrt_race_path = drone_racing_rrt(start_demo2, targets_demo2)
print(f"RRT path: {len(rrt_race_path)} waypoints")

print("\nStep 2: Optimizing racing trajectory...")
optimized_race, success_race, info_race = helpers.optimize_racing_path_sequential(
    rrt_path=rrt_race_path,
    start_pose=start_demo2,
    hoops=hoops_demo2,
    obstacles=[],
    N=25,
    dt=0.1,
    weights={'thrust': 0.05, 'angular': 0.5, 'smoothness': 5.0},
    # Pass student-implemented functions (TODOs 17-28)
    initialize_from_rrt_robust=initialize_from_rrt_robust,
    dynamics_constraints_robust=dynamics_constraints_robust,
    boundary_constraints_robust=boundary_constraints_robust,
    collision_constraints_optimized=collision_constraints_optimized,
    cost_function_tuned=cost_function_tuned,
    unpack_decision_vars=unpack_decision_vars
)

if success_race:
    print(f"\n✅ Racing optimization SUCCESS!")
else:
    print(f"\n⚠ Some segments may have failed")


print("\nStep 3: Visualizing...")

# Visualize racing comparison
fig = helpers.drone_racing_path_comparison(
    hoops_demo2, start_demo2, rrt_race_path, optimized_race,
    title="Demo 2: RRT vs Optimized Racing"
)
fig.show()


### 🏎️ Racing Analysis

**RRT (cyan)**: Gets through hoops but with sharp turns
**Optimized (magenta)**: Smooth arcing turns, consistent velocity

The optimized path is:
- Safer (lower peak accelerations)
- More efficient (better battery life)
- Easier to track (predictable for controllers)

Real drone racing teams use exactly this approach!


### 7.3.3 Velocity & Acceleration Profiles
Let's look "under the hood" at the velocity and acceleration.


In [ ]:
if success_race and 'states' in info_race:
    states_opt = info_race['states']
    controls_opt = info_race['controls']

    # Compute velocities
    dt = 0.1
    velocities = np.diff(states_opt[:, :3], axis=0) / dt
    accelerations = np.diff(velocities, axis=0) / dt

    speeds = np.linalg.norm(velocities, axis=1)
    print(f"Velocity stats:")
    print(f"   Max speed: {speeds.max():.2f} m/s")
    print(f"   Mean speed: {speeds.mean():.2f} m/s")

    accel_mags = np.linalg.norm(accelerations, axis=1)
    print(f"\nAcceleration stats:")
    print(f"   Max: {accel_mags.max():.2f} m/s²")
    print(f"   Mean: {accel_mags.mean():.2f} m/s²")

    print(f"\nControl stats:")
    print(f"   Thrust range: [{controls_opt[:, 3].min():.1f}, {controls_opt[:, 3].max():.1f}]")
    print(f"   Mean thrust: {controls_opt[:, 3].mean():.1f} (hover=10.0)")

    # Plot
    fig = helpers.plot_velocity_acceleration_profiles(velocities, accelerations, controls_opt, dt)
    fig.show()
else:
    print("⚠ Profile data not available")


### 7.3.4 Path Metric Comparison
Let's quantify the improvement on how our optimized path is better than the non-optimized one!

In [ ]:
def compute_path_length(path):
    """Compute total path length"""
    length = 0.0
    for i in range(len(path)-1):
        pos1 = np.array(path[i].translation())
        pos2 = np.array(path[i+1].translation())
        length += np.linalg.norm(pos2 - pos1)
    return length

length_rrt = compute_path_length(rrt_race_path)
length_opt = compute_path_length(optimized_race)

print("PATH COMPARISON:")
print(f"  RRT path length: {length_rrt:.2f} m")
print(f"  Optimized path length: {length_opt:.2f} m")
print(f"  Reduction: {(length_rrt-length_opt)/length_rrt*100:.1f}%")
print(f"\n  RRT waypoints: {len(rrt_race_path)}")
print(f"  Optimized waypoints: {len(optimized_race)}")

if success_race:
    print(f"\n✨ The optimized path is smoother and more efficient!")


## 🏁 PART 8: The Final Challenge: Racing with Obstacles!

**This is it.** Everything you've built - RRT planning, terminal velocity dynamics, trajectory optimization, constraint handling - all comes together RIGHT NOW.

You've seen your drone fly smooth paths. You've watched it race through hoops. But now? Now we add **obstacles** to the mix. This is the ultimate test of your optimizer.

### What You're About to Do:

1. **Navigate through 4 aerial hoops** (you've done this before!)
2. **Dodge obstacles** placed strategically in your path (NEW!)
3. **Satisfy ALL constraints simultaneously:**
   - Dynamics constraints (physics must be obeyed)
   - Boundary constraints (start and goal positions)
   - Hoop constraints (pass through centers)
   - Collision constraints (avoid obstacles with safety margin)
4. **Optimize for smoothness** while doing all of the above

### Why This Matters:

This isn't just a demo anymore. This is **real autonomous drone racing**. The kind of problem that gets drones through disaster zones, warehouse navigation, and yes - actual competitive racing leagues.

Your RRT will plan a collision-free path. Your optimizer will make it fast and smooth. Your constraints will keep it safe and accurate.

### The Stakes:

- If your dynamics constraints fail -> your drone violates physics
- If your boundary constraints fail -> you miss the start or goal
- If your collision constraints fail -> you crash into obstacles
- If your optimization succeeds → **YOU WIN** 🏆

Take a deep breath. Run the cells below. Watch your drone navigate the obstacle course like a boss.

**Let's do this!** 🚀


In [ ]:
print("\n" + "="*80)
print("🏁 FINAL CHALLENGE: RACING WITH OBSTACLES (EASY)")
print("="*80 + "\n")

# Setup
hoops_final = helpers.get_hoops()
targets_final = helpers.get_targets()
start_final = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(1, 3, 8))
obstacles_easy = helpers.get_obstacles_easy()

print(f"Configuration:")
print(f"   - Start: {start_final.translation()}")
print(f"   - Hoops: {len(hoops_final)}")
print(f"   - Obstacles: {len(obstacles_easy)} (EASY)")

print("\n" + "-"*80)
print("STAGE 1: RRT WITH OBSTACLES")
print("-"*80)

import time
start_time = time.time()
rrt_final_path = drone_racing_rrt_with_obstacles(start_final, targets_final, obstacles_easy)
rrt_time = time.time() - start_time

print(f"RRT completed in {rrt_time:.2f}s")
print(f"Path: {len(rrt_final_path)} waypoints")

has_collision, _ = helpers.check_path_collision(rrt_final_path, obstacles_easy)
if not has_collision:
    print(f"RRT is Collision-free!")
else:
    print(f"RRT may have collisions")


In [ ]:
# Visualize RRT racing path with obstacles once again
helpers.drone_racing_path_with_obstacles(
    hoops=helpers.get_hoops(),
    start=start_final,
    path=rrt_final_path,
    obstacles=obstacles_easy
)

print(f"\nTotal racing path length: {len(rrt_final_path)} waypoints")

# Verify collision-free
has_collision, _ = helpers.check_path_collision(rrt_final_path, obstacles_easy)
if not has_collision:
    print(f"RRT is Collision-free!")
else:
    print(f"RRT may have collisions")

In [ ]:
print("\n" + "-"*80)
print("STAGE 2: TRAJECTORY OPTIMIZATION")
print("-"*80)

opt_start = time.time()
optimized_final, success_final, info_final = helpers.optimize_racing_path_sequential(
    rrt_path=rrt_final_path,
    start_pose=start_final,
    hoops=hoops_final,
    obstacles=obstacles_easy,
    N=25,
    dt=0.1,
    weights={'thrust': 0.05, 'angular': 0.5, 'smoothness': 5.0},
    initialize_from_rrt_robust=initialize_from_rrt_robust,
    dynamics_constraints_robust=dynamics_constraints_robust,
    boundary_constraints_robust=boundary_constraints_robust,
    collision_constraints_optimized=collision_constraints_optimized,
    cost_function_tuned=cost_function_tuned,
    unpack_decision_vars=unpack_decision_vars
)
opt_time = time.time() - opt_start

if success_final:
    print(f"\n✅ OPTIMIZATION SUCCESS!")
    print(f"   Time: {opt_time:.2f}s")
    print(f"   Total time: {rrt_time + opt_time:.2f}s")

    has_collision_opt, _ = helpers.check_path_collision(optimized_final, obstacles_easy)
    if not has_collision_opt:
        print(f"Optimized path is collision-free!")
    print(f"Passes through all {len(hoops_final)} hoops")
else:
    print(f"\n Optimization had issues, using RRT path")
    optimized_final = rrt_final_path


In [ ]:
print("\n" + "-"*80)
print("STAGE 3: VISUALIZATION")
print("-"*80)

helpers.drone_racing_path_with_obstacles(
    hoops_final, start_final, optimized_final, obstacles_easy,
)

print("\n✨ Interactive 3D visualization above!")
print("   - Rotate: mouse drag")
print("   - Zoom: scroll wheel")
print("   - Pan: right-click drag")

In [ ]:
# Path Metric Evaluation:
length_rrt_final = compute_path_length(rrt_final_path)
length_opt_final = compute_path_length(optimized_final)

print("PATH COMPARISON:")
print(f"  RRT path length: {length_rrt:.2f} m")
print(f"  Optimized path length: {length_opt:.2f} m")
print(f"  Reduction: {(length_rrt-length_opt)/length_rrt*100:.1f}%")
print(f"\n  RRT waypoints: {len(rrt_race_path)}")
print(f"  Optimized waypoints: {len(optimized_race)}")

if success_race:
    print(f"\n✨ The optimized path is smoother and more efficient!")

## Final Reflection Questions:

#### Part A: Optimization Sensitivity Analysis
Similar to how we analyzed drone dynamics earlier, let's analyze how our optimization hyperparameters affect the final trajectory. Consider the following scenarios for the trajectory optimization. For each case, hypothesize (or test!) what will happen to the **path shape**, **speed**, and **computation time**.

| Scenario | `w_smoothness` | `w_thrust` | `N` (Knot Points) | Description |
| :--- | :---: | :---: | :---: | :--- |
| **1. Baseline** | 5.0 | 0.05 | 25 | Balanced parameters |
| **2. Ultra Smooth** | 50.0 | 0.05 | 25 | Very high smoothness penalty |
| **3. Aggressive** | 0.1 | 1.0 | 25 | Low smoothness, high thrust cost |
| **4. Low Res** | 5.0 | 0.05 | 10 | Very few knot points |
| **5. High Res** | 5.0 | 0.05 | 100 | Many knot points |

**Questions**
1. **Smoothness vs. Feasibility:** In Scenario 2 (Ultra Smooth), what happens to the drone's ability to pass through narrow hoops or avoid obstacles? Does the path become "too" simple?
2. **Aggressiveness:** In Scenario 3, does the drone fly faster or slower? Does it cut corners more tightly? What happens to the control inputs (thrust/body rates)?
3. **Resolution Trade-off:** Compare Scenarios 4 and 5. How does `N` affect the *validity* of the trajectory (collision checking resolution) vs. the *optimality* and *computation time*? Why can't we just use N=1000 for everything?

#### Part B: Overall Reflection
1. **RRT vs. Optimization:** How do these two approaches complement each other? Why is RRT good for initialization but bad for final execution? Why is Optimization good for smoothing but bad for finding a path from scratch?
2. **The "Reality Gap":** You implemented a physics-based model ($F=ma$, drag, etc.). In the real world, what other factors (wind, battery voltage sag, sensor noise, communication delay) would make this difficult? How would you modify your controller to handle them?
3. **Course Feedback:** What was the most "Aha!" moment for you in this assignment?

## Response:
###Part A
1. Smoothness v/s Feasibility:  
- There will be a bad effect of passing through the narrow hoops and while avoidng obstacles. This is because the increase in smoothness will result in proioritizing the minimization of change in acceleration than completion of the task. So when a sharp turn is required, it will still try and take a longer shoother turn, at times increasing the time to complete the task. As a result if there are hoops consecutively at short steep angles, it might fail to manuver them or take multiple tries or longer paths to go through them.
- Yes, the path becomes "too" simple. High penalty forces it to take the smoother turns and not the sharp turns. This makes it impossible to transit through sharp turns and low clearance paths which might ultimately result in failure.

2. Aggressiveness:
- The quadrotor will actually fly slower because of higher penalty on use of thrust. It will now try to cover as much path as possible with the same speed instead of increasing the speed. As the result the quadrotor will have a gliding-like flight.
-Yes it will now take sharper turns since the smoothness-penalty is very less. The path will have long straight paths with acute turns. This is because there is penalty for smoothness is very less and high for thrust.

3. Resolution Trade-off:
- With low resolution, the trajactory becomes very coarse, like connecting dots with straight lines. Validiy issues include collision checking which it might miss between the knots and even skip over hoop centers but it will have very fast computation. With high resolution, there will be excellent collision detection with high computation time. It might not even converge over the limited iterations.
We cannot just use N=1000 since it would mean nearly 10000 variables! It would take hours to complete, increases the computation complexity but at the same time give exceptional quality and collision detection.


###Part B
1. RRT vs Optimization:
Initially I thought why we were using both but then the results made the difference clear. RRT basically finds the rough solution and is thus very fast to find the path. On the other hand optimization helps fidning the best from that path by improving the turns, maintaining max velocity and achieving the goal in least amount of time possible.

2. The reality gap:
Our simulation had gravity, drag, rotation limits, thrust bounds. But real flying is way harder.
Wind constantly pushes the drone off course. Turbulence near buildings throws it around unpredictably. Batteries die - start with 20N max thrust, end with barely 10N (hover only). Everything lags - motors take 50-100ms to respond, sensors are noisy and delayed.
To fly for real, I'd add: sensor fusion for noisy data, MPC replanning every 0.1s instead of blindly following the path, wind estimation to compensate gusts, battery monitoring to adjust thrust limits, and a fast 100Hz feedback loop with PID corrections. These are some of the moderations I made during my path planning project and I think these would be relevant here as well.

3. Aha Moments:
There were quite a few Aha moments! Be it after solving an almost non existant bug or seeing the simulation run for the first time. It really ws an Aha moment to see how building such an algorithm stepwise from scratch and see it successfully navigate from one point to another, across the hoops while avoiding the with objects. Really loved working on this assignment.

# Scenerio 2

In [ ]:
print("\n" + "-"*80)
print("STAGE 2: TRAJECTORY OPTIMIZATION")
print("-"*80)

opt_start = time.time()
optimized_final, success_final, info_final = helpers.optimize_racing_path_sequential(
    rrt_path=rrt_final_path,
    start_pose=start_final,
    hoops=hoops_final,
    obstacles=obstacles_easy,
    N=25,
    dt=0.1,
    weights={'thrust': 0.05, 'angular': 0.5, 'smoothness': 50.0},
    initialize_from_rrt_robust=initialize_from_rrt_robust,
    dynamics_constraints_robust=dynamics_constraints_robust,
    boundary_constraints_robust=boundary_constraints_robust,
    collision_constraints_optimized=collision_constraints_optimized,
    cost_function_tuned=cost_function_tuned,
    unpack_decision_vars=unpack_decision_vars
)
opt_time = time.time() - opt_start

if success_final:
    print(f"\n✅ OPTIMIZATION SUCCESS!")
    print(f"   Time: {opt_time:.2f}s")
    print(f"   Total time: {rrt_time + opt_time:.2f}s")

    has_collision_opt, _ = helpers.check_path_collision(optimized_final, obstacles_easy)
    if not has_collision_opt:
        print(f"Optimized path is collision-free!")
    print(f"Passes through all {len(hoops_final)} hoops")
else:
    print(f"\n Optimization had issues, using RRT path")
    optimized_final = rrt_final_path


print("\n" + "-"*80)
print("STAGE 3: VISUALIZATION")
print("-"*80)

helpers.drone_racing_path_with_obstacles(
    hoops_final, start_final, optimized_final, obstacles_easy,
)

print("\n✨ Interactive 3D visualization above!")
print("   - Rotate: mouse drag")
print("   - Zoom: scroll wheel")
print("   - Pan: right-click drag")


# Scenerio 3

In [ ]:
print("\n" + "-"*80)
print("STAGE 2: TRAJECTORY OPTIMIZATION")
print("-"*80)

opt_start = time.time()
optimized_final, success_final, info_final = helpers.optimize_racing_path_sequential(
    rrt_path=rrt_final_path,
    start_pose=start_final,
    hoops=hoops_final,
    obstacles=obstacles_easy,
    N=25,
    dt=0.1,
    weights={'thrust': 1.0, 'angular': 0.5, 'smoothness': 0.1},
    initialize_from_rrt_robust=initialize_from_rrt_robust,
    dynamics_constraints_robust=dynamics_constraints_robust,
    boundary_constraints_robust=boundary_constraints_robust,
    collision_constraints_optimized=collision_constraints_optimized,
    cost_function_tuned=cost_function_tuned,
    unpack_decision_vars=unpack_decision_vars
)
opt_time = time.time() - opt_start

if success_final:
    print(f"\n✅ OPTIMIZATION SUCCESS!")
    print(f"   Time: {opt_time:.2f}s")
    print(f"   Total time: {rrt_time + opt_time:.2f}s")

    has_collision_opt, _ = helpers.check_path_collision(optimized_final, obstacles_easy)
    if not has_collision_opt:
        print(f"Optimized path is collision-free!")
    print(f"Passes through all {len(hoops_final)} hoops")
else:
    print(f"\n Optimization had issues, using RRT path")
    optimized_final = rrt_final_path


print("\n" + "-"*80)
print("STAGE 3: VISUALIZATION")
print("-"*80)

helpers.drone_racing_path_with_obstacles(
    hoops_final, start_final, optimized_final, obstacles_easy,
)

print("\n✨ Interactive 3D visualization above!")
print("   - Rotate: mouse drag")
print("   - Zoom: scroll wheel")
print("   - Pan: right-click drag")


# Scenerio 4

In [ ]:
print("\n" + "-"*80)
print("STAGE 2: TRAJECTORY OPTIMIZATION")
print("-"*80)

opt_start = time.time()
optimized_final, success_final, info_final = helpers.optimize_racing_path_sequential(
    rrt_path=rrt_final_path,
    start_pose=start_final,
    hoops=hoops_final,
    obstacles=obstacles_easy,
    N=10,
    dt=0.1,
    weights={'thrust': 0.05, 'angular': 0.5, 'smoothness': 5.0},
    initialize_from_rrt_robust=initialize_from_rrt_robust,
    dynamics_constraints_robust=dynamics_constraints_robust,
    boundary_constraints_robust=boundary_constraints_robust,
    collision_constraints_optimized=collision_constraints_optimized,
    cost_function_tuned=cost_function_tuned,
    unpack_decision_vars=unpack_decision_vars
)
opt_time = time.time() - opt_start

if success_final:
    print(f"\n✅ OPTIMIZATION SUCCESS!")
    print(f"   Time: {opt_time:.2f}s")
    print(f"   Total time: {rrt_time + opt_time:.2f}s")

    has_collision_opt, _ = helpers.check_path_collision(optimized_final, obstacles_easy)
    if not has_collision_opt:
        print(f"Optimized path is collision-free!")
    print(f"Passes through all {len(hoops_final)} hoops")
else:
    print(f"\n Optimization had issues, using RRT path")
    optimized_final = rrt_final_path


print("\n" + "-"*80)
print("STAGE 3: VISUALIZATION")
print("-"*80)

helpers.drone_racing_path_with_obstacles(
    hoops_final, start_final, optimized_final, obstacles_easy,
)

print("\n✨ Interactive 3D visualization above!")
print("   - Rotate: mouse drag")
print("   - Zoom: scroll wheel")
print("   - Pan: right-click drag")


# Scenerio 5

In [ ]:
print("\n" + "-"*80)
print("STAGE 2: TRAJECTORY OPTIMIZATION")
print("-"*80)

opt_start = time.time()
optimized_final, success_final, info_final = helpers.optimize_racing_path_sequential(
    rrt_path=rrt_final_path,
    start_pose=start_final,
    hoops=hoops_final,
    obstacles=obstacles_easy,
    N=100,
    dt=0.1,
    weights={'thrust': 0.05, 'angular': 0.5, 'smoothness': 5.0},
    initialize_from_rrt_robust=initialize_from_rrt_robust,
    dynamics_constraints_robust=dynamics_constraints_robust,
    boundary_constraints_robust=boundary_constraints_robust,
    collision_constraints_optimized=collision_constraints_optimized,
    cost_function_tuned=cost_function_tuned,
    unpack_decision_vars=unpack_decision_vars
)
opt_time = time.time() - opt_start

if success_final:
    print(f"\n✅ OPTIMIZATION SUCCESS!")
    print(f"   Time: {opt_time:.2f}s")
    print(f"   Total time: {rrt_time + opt_time:.2f}s")

    has_collision_opt, _ = helpers.check_path_collision(optimized_final, obstacles_easy)
    if not has_collision_opt:
        print(f"Optimized path is collision-free!")
    print(f"Passes through all {len(hoops_final)} hoops")
else:
    print(f"\n Optimization had issues, using RRT path")
    optimized_final = rrt_final_path


print("\n" + "-"*80)
print("STAGE 3: VISUALIZATION")
print("-"*80)

helpers.drone_racing_path_with_obstacles(
    hoops_final, start_final, optimized_final, obstacles_easy,
)

print("\n✨ Interactive 3D visualization above!")
print("   - Rotate: mouse drag")
print("   - Zoom: scroll wheel")
print("   - Pan: right-click drag")


### 🎉 Victory!
<div>
<img src="https://drive.google.com/uc?export=view&id=1E6KVv2vXcpFsMoCGG_u1eNNAtsAai6Ra" width="500"/>
</div>

**This is the end of the assignmnet. Everything after this point is extra credit. (Extra credit will be updated shortly)**

Your drone just navigated through hoops AND dodged obstacles. Let that sink in for a second.

**This is the kind of tech that:**
- Powers Amazon delivery drones navigating urban environments
- Enables search-and-rescue drones in disaster zones
- Wins actual drone racing competitions (DRL, MultiGP)
- Gets used in Hollywood for autonomous aerial cinematography
- Lands rovers on Mars (okay, different dynamics, but same optimization principles!)


---
---

### 💪 What You Can Do Now:

- **Impress recruiters:** "I implemented nonlinear trajectory optimization with collision avoidance for autonomous drones"
- **Side projects:** Hook this up to a real drone (Tello EDU is like $100)
- **Research:** Extend it to multi-drone coordination, time-optimal trajectories, or learning-based planning
- **Competitions:** Seriously, there are drone racing leagues that need exactly this kind of work but with better planners!

---

### 🔥 Hungry for More?

If you're the kind of person who thinks "that was too easy" (it wasn't, but we respect the confidence), scroll down.

**The EXTRA CREDIT challenge below will separate the good from the legendary.** 👇

### 🌟 EXTRA CREDIT: The Gauntlet

#### The Challenge That Breaks Most Students

The easy course? That was a tutorial. **This is the main level.**

#### What You're Up Against:

**The HARD Obstacle Course:**
- **8 obstacles** (4x more than before)
- **Narrow gaps** between obstacles (your safety margin will be tested)
- **Complex 3D geometry** (boxes AND spheres, positioned to trap bad planners)
- **Longer computation** (RRT might need multiple attempts)
- **Tighter optimization** (SLSQP might fail, trust-constr will earn its keep)

#### You Have Total Freedom:

Unlike the main assignment, **you can use ANY path planning algorithm you want:**
- Stick with RRT (the classic)
- Implement RRT* (asymptotically optimal)
- Try RRT-Connect (bidirectional search)
- Use informed RRT* (if you're showing off)
- Implement MPPI, A*, DWA (if you really have that much time!)
- Design a custom heuristic planner
- Combine multiple strategies
- **OR Surprise us!**

The only rule: **Don't hardcode the solution.** Your planner must work for arbitrary obstacle configurations.

#### ⚠️ The Box Obstacle Twist:

Here's the kicker: **The easy course only had spherical obstacles.** Your `collision_constraints_optimized` function (TODO 26) probably only handles sphere collision checks.

**The hard course has BOX obstacles too.**

To complete this challenge, you need to:
1. **Extend `collision_constraints_optimized`** to handle both sphere AND box obstacles
2. **Implement box collision constraints** (hint: think about axis-aligned bounding boxes—check if drone position is within [min_corner, max_corner] plus safety margin)
3. **Make it work seamlessly** with your existing optimizer

**Define the `collision_constraints_optimized` or any new functions again which you want to modify from the previous sections (don't modify them in place as it can hinder your assignment grades)**

#### 🏆 Grading Rubric (Worth It):

**Core Challenge (Multiple Obstacles):**
- ✅ **+10%** Implement box collision constraints accurately
- ✅ **+5%** Optimization converges (constraints satisfied)
- ✅ **+10%** Final optimized path avoids ALL obstacles (spheres + boxes)

**Bonus Points (Flex Zone):**
- ✅ **+20%** Use a non-RRT planner successfully (RRT*, MPPI, etc.) which avoids all obstacles
- ✅ **+5%** Document your approach with detailed comments/markdown

**Total Possible: 50% of extra credit value**

#### Extra Credit Grading Scale:

- **Assignment base**: 100 points
- **Extra credit raw**: Up to 50 points (percentages above)
- **How it works**: Your extra credit percentage is scaled proportionally to contribute up to bonus of upto half of the total assignment marks

**Bottom line:** Every bit counts, and perfection here can seriously boost your grade!



In [ ]:
# Before Commiting for extra credit, let's see how the whole Racing environment looks like with 8 obstacles added instead of 2
start_race = gtsam.Pose3(r=gtsam.Rot3.Yaw(math.radians(45)), t=gtsam.Point3(1, 3, 8))
helpers.drone_racing_path_with_obstacles(helpers.get_hoops(), start_race, [], obstacles=helpers.get_obstacles_hard())

#### 💀 Why This Is Hard:

- **Box obstacles are NEW** -> Your collision function only handles spheres right now
- **RRT might fail** -> You'll need retry logic or adaptive parameters
- **Optimization might fail** -> You'll need good initialization
- **Collisions are sneaky** -> Subsampling must be dense enough, especially for boxes
- **Constraints conflict** -> Hoops might be near obstacles
- **Mixed geometry** -> Handling both sphere AND box constraints simultaneously
- **Computation time** -> You might wait 2-5 minutes per attempt

#### 💡 Hints (Use Wisely):

1. **Box collision math:** For a point to be OUTSIDE a box, it must be outside on at least one face. Constraint per box: `max(0, -(pos[0] - box.max[0] - safety), -(box.min[0] - pos[0] - safety), ...)`
2. **RRT struggles?** Try increasing max iterations or adding goal bias
3. **Optimization fails?** Use more knot points (N=40-50) or adjust weights
4. **Collisions after optimization?** Check your `collision_constraints_optimized` function handles BOTH obstacle types
5. **Getting desperate?** The provided code has retry logic for RRT (see below)

#### 🎯 The Mindset:

This is **research-level** robotics. Real autonomous systems fail sometimes. Your job is to:
1. **Understand why** things fail
2. **Adapt your strategy**
3. **Persist until it works**

If you get this working, you're in the top 5% of students. If you get bonus points, you're ready for graduate-level robotics.

**No pressure. Just glory.** 🔥


In [ ]:
print("\n" + "="*80)
print("🌟 EXTRA CREDIT: HARD OBSTACLE COURSE")
print("="*80 + "\n")

# Setup HARD course
hoops_hard = helpers.get_hoops()
targets_hard = helpers.get_targets()
start_hard = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(1, 3, 8))
obstacles_hard = helpers.get_obstacles_hard()

print(f"⚠️ WARNING: HARD COURSE! PROCEED AT YOUR OWN RISK ⚠️")
print(f"   - Hoops: {len(hoops_hard)}")
print(f"   - Obstacles: {len(obstacles_hard)} (vs {len(obstacles_easy)} in easy)")
print(f"   - Expected time: 30-90s")

# You may need to tweak RRT parameters for success here

print("\n" + "-"*80)
print("STAGE 1: RRT WITH HARD OBSTACLES")
print("-"*80)

# Try RRT multiple times
rrt_hard_success = False
max_attempts = 3

for attempt in range(1, max_attempts + 1):
    print(f"\nAttempt {attempt}/{max_attempts}...")
    try:
        start_time_hard = time.time()
        rrt_hard_path = drone_racing_rrt_with_obstacles(start_hard, targets_hard, obstacles_hard)
        rrt_hard_time = time.time() - start_time_hard

        has_collision_rrt, _ = helpers.check_path_collision(rrt_hard_path, obstacles_hard)

        if not has_collision_rrt and len(rrt_hard_path) > 0:
            print(f"✓ SUCCESS on attempt {attempt}!")
            print(f"✓ Time: {rrt_hard_time:.2f}s")
            print(f"✓ Path: {len(rrt_hard_path)} waypoints")
            rrt_hard_success = True
            break
        else:
            print(f"❌ Attempt {attempt} failed")
    except Exception as e:
        print(f"❌ Error: {e}")

if not rrt_hard_success:
    print(f"\n⚠️ RRT failed after {max_attempts} attempts")
    print(f"   Suggestions:")
    print(f"   - Increase RRT threshold")
    print(f"   - Add goal biasing")
    print(f"   - Increase max iterations")


if rrt_hard_success:
    print("\n" + "-"*80)
    print("STAGE 2: OPTIMIZATION (HARD)")
    print("-"*80)

    opt_hard_start = time.time()
    optimized_hard, success_hard, info_hard = helpers.optimize_racing_path_sequential(
        rrt_path=rrt_hard_path,
        start_pose=start_hard,
        hoops=hoops_hard,
        obstacles=obstacles_hard,
        N=30,
        dt=0.1,
        weights={'thrust': 0.03, 'angular': 0.3, 'smoothness': 8.0},
        # Pass student-implemented functions (TODOs 17-28)
        initialize_from_rrt_robust=initialize_from_rrt_robust,
        dynamics_constraints_robust=dynamics_constraints_robust,
        boundary_constraints_robust=boundary_constraints_robust,
        collision_constraints_optimized=collision_constraints_optimized,
        cost_function_tuned=cost_function_tuned,
        unpack_decision_vars=unpack_decision_vars
    )
    opt_hard_time = time.time() - opt_hard_start

    if success_hard:
        print(f"\n🎉 HARD COURSE SUCCESS! 🎉")
        print(f"   Time: {opt_hard_time:.2f}s")

        has_collision_hard, _ = helpers.check_path_collision(optimized_hard, obstacles_hard)

        print(f"\n🏆 EXTRA CREDIT CHECKLIST:")
        print(f"   [✅] RRT succeeded")
        print(f"   [{'✅' if success_hard else '❌'}] Optimization converged")
        print(f"   [{'✅' if not has_collision_hard else '❌'}] Collision-free")
        print(f"   [✅] Through all hoops")

        if success_hard and not has_collision_hard:
            print(f"\n   🌟🌟🌟 EXTRA CREDIT Earned! Implement a Custom Planner Now! 🌟🌟🌟")
        else:
            print(f"\n   💪 Good effort! Try tuning weights")
    else:
        print(f"\n❌ Optimization failed for hard course")
        print(f"   Partial credit for RRT success!")
        optimized_hard = rrt_hard_path
else:
    print("\n⏭️ Skipping optimization (RRT failed)")

if rrt_hard_success:
    print("\n" + "-"*80)
    print("STAGE 3: VISUALIZATION (HARD)")
    print("-"*80)

    fig_hard = helpers.drone_racing_path_with_obstacles(
        hoops_hard, start_hard,
        optimized_hard if success_hard else rrt_hard_path,
        obstacles_hard,
        title="🌟 Extra Credit: Hard Obstacle Course"
    )
    fig_hard.show()

    print("\n✨ Notice how challenging this environment is!")


In [ ]:
#### Student implements Custom planner and optimization here ####

#### End student code ####

# Extra Credit Code
This is the code for the extra credits, that is Part 8 of the Coding Assessment 2. Here I am using A* algorithm for finding a path from initial given pose3 to the target pose3.

A* Algorithm: A* algorithm divides the whole 3D envirnment into grids. It then marks all the grid cells that have the obstacle in them. Finally by defining the cost function, it tries to reach the goal position with the least cost.

In [ ]:
## installation

## imports
from typing import List, Tuple, Dict, Optional, Set
import math
import numpy as np
import pandas as pd
import unittest
import plotly.express as px
import plotly.graph_objects as go
%pip install gtsam==4.3a0
import gtsam
import time
import heapq

rng = np.random.default_rng(12345)


### A* Configuration Class
This class defines the parameters we will use for the A* Algorithm.

In [ ]:
# A* Planner Configuration Class

class AStarConfig:
    #Configuration parameters for A* path planner

    # Grid settings
    GRID_RESOLUTION = 0.4  # meters per cell

    # Workspace bounds [min, max] for each axis
    WORKSPACE_X = (0.0, 10.0)
    WORKSPACE_Y = (0.0, 10.0)
    WORKSPACE_Z = (0.0, 10.0)

    # Safety margin around obstacles
    SAFETY_MARGIN = 0.2  # meters (keep drone 0.2m away from obstacles)

    # Movement costs
    COST_STRAIGHT = 1.0             # Moving along one axis
    COST_DIAGONAL_2D = np.sqrt(2)   # Moving along two axes
    COST_DIAGONAL_3D = np.sqrt(3)   # Moving along three axes

    # Neighbor connectivity
    USE_26_CONNECTED = True   # 3x3x3 = 27, Subtracting 1 (Current position)

    def __init__(self):
        # Calculate grid dimensions
        # Adding +1 to ensure we can reach max boundary
        self.nx = int(np.ceil((self.WORKSPACE_X[1] - self.WORKSPACE_X[0]) / self.GRID_RESOLUTION))
        self.ny = int(np.ceil((self.WORKSPACE_Y[1] - self.WORKSPACE_Y[0]) / self.GRID_RESOLUTION))
        self.nz = int(np.ceil((self.WORKSPACE_Z[1] - self.WORKSPACE_Z[0]) / self.GRID_RESOLUTION))

        print(f"Grid Configuration:")
        print(f"  Resolution: {self.GRID_RESOLUTION}m")
        print(f"  Dimensions: {self.nx} x {self.ny} x {self.nz} = {self.nx * self.ny * self.nz:,} cells")
        print(f"  Safety margin: {self.SAFETY_MARGIN}m")
        print(f"  Connectivity: {'26-connected' if self.USE_26_CONNECTED else '6-connected'}")

# Create configuration instance
config = AStarConfig()

### Grid Cell Class
This class gives the parameters for the grids that we divide our envirnment in!

In [ ]:
# Grid Cell Class - Represents a single cell in the 3D grid

class GridCell:
    """
    Represents a single cell in the 3D A* search grid

    Attributes:
        i, j, k: Grid coordinates (integers)
        x, y, z: World coordinates (floats) - center of cell
        g_cost: Actual cost from start to this cell
        h_cost: Heuristic estimate from this cell to goal
        f_cost: Total cost (g + h)
        parent: Parent GridCell (for path reconstruction)
        is_obstacle: True if cell contains obstacle
    """

    def __init__(self, i: int, j: int, k: int, world_x: float, world_y: float, world_z: float):
        # Grid coordinates
        self.i = i
        self.j = j
        self.k = k

        # World coordinates
        self.x = world_x
        self.y = world_y
        self.z = world_z

        # A* costs
        self.g_cost = float('inf')
        self.h_cost = 0.0
        self.f_cost = float('inf')

        # Path reconstruction
        self.parent = None

        # Obstacle flag
        self.is_obstacle = False

    def __lt__(self, other):
        # For priority queue comparison - compare f_costs
        return self.f_cost < other.f_cost

    def __eq__(self, other):
        # Two cells are equal if they have same grid coordinates
        return (self.i == other.i and
                self.j == other.j and
                self.k == other.k)

    def __hash__(self):
        # For use in sets and dictionaries
        return hash((self.i, self.j, self.k))

    def position_tuple(self):
        # Return (i, j, k) as tuple for easy comparison
        return (self.i, self.j, self.k)

    def world_position(self):
        # Return (x, y, z) as numpy array
        return np.array([self.x, self.y, self.z])

    def __repr__(self):
        return f"Cell({self.i},{self.j},{self.k}) @ ({self.x:.1f},{self.y:.1f},{self.z:.1f})"


print("Grid Cell class created!")


### Frame Conversion Functions
These functions are used to convert the coordinates from given frame to the A* algorithm's grid frame. Basically the world frame has 1m as unit length, but grid has 0.4m as the unit distance.

In [ ]:
# Coordinate conversion functions

def world_to_grid(world_pos: np.ndarray, config: AStarConfig) -> Tuple[int, int, int]:
    """
    Convert world coordinates (x, y, z) to grid indices (i, j, k)

    Arguments:
        world_pos: numpy array [x, y, z] in meters
        config: AStarConfig with grid parameters

    Returns:
        (i, j, k): Grid cell indices (integers)

    """

    # Extract x, y, z from world_pos
    x, y, z = world_pos[0], world_pos[1], world_pos[2]

    # Convert to grid indices
    # i = (x - x_min) / resolution, rounded down
    i = int((x - config.WORKSPACE_X[0]) / config.GRID_RESOLUTION)
    j = int((y - config.WORKSPACE_Y[0]) / config.GRID_RESOLUTION)
    k = int((z - config.WORKSPACE_Z[0]) / config.GRID_RESOLUTION)

    # Clamp to valid range [0, n-1] to prevent out-of-bounds errors
    i = np.clip(i, 0, config.nx - 1)
    j = np.clip(j, 0, config.ny - 1)
    k = np.clip(k, 0, config.nz - 1)
    return (i, j, k)


def grid_to_world(i: int, j: int, k: int, config: AStarConfig) -> np.ndarray:
    """
    Convert grid indices (i, j, k) to world coordinates (x, y, z)
    Returns the CENTER of the cell

    Arguments:
        i, j, k: Grid cell indices (integers)
        config: AStarConfig with grid parameters

    Returns:
        world_pos: numpy array [x, y, z] at cell center

    """

    # Add 0.5 to get cell center, then multiply by resolution
    x = config.WORKSPACE_X[0] + (i + 0.5) * config.GRID_RESOLUTION
    y = config.WORKSPACE_Y[0] + (j + 0.5) * config.GRID_RESOLUTION
    z = config.WORKSPACE_Z[0] + (k + 0.5) * config.GRID_RESOLUTION

    world_pos = np.array([x, y, z])

    return world_pos


def is_valid_cell(i: int, j: int, k: int, config: AStarConfig) -> bool:
    """
    Check if grid indices are within valid bounds

    Arguments:
        i, j, k: Grid cell indices
        config: AStarConfig with grid dimensions

    Returns:
        True if cell is within grid bounds, False otherwise
    """

    valid = (0 <= i < config.nx and
             0 <= j < config.ny and
             0 <= k < config.nz)

    return valid


#### Example of how the convertion functions work.

In [ ]:
# Test the conversion functions
print("Testing coordinate conversions...")

test_world = np.array([2.5, 3.7, 6.1])
test_grid = world_to_grid(test_world, config)
print(f"World {test_world} → Grid {test_grid}")

test_world_back = grid_to_world(test_grid[0], test_grid[1], test_grid[2], config)
print(f"Grid {test_grid} → World {test_world_back}")

print("\nCoordinate conversion functions working!")

### Collision Detection Fucntions for Sphere and Box Obstacles


In [ ]:
# Sphere obstacle collision detection

def is_cell_in_sphere_obstacle(cell_world_pos: np.ndarray, sphere_center: np.ndarray, sphere_radius: float, safety_margin: float) -> bool:
    """
    Check if a cell (at its center) is inside a sphere obstacle (with safety margin)

    Arguments:
        cell_world_pos: numpy array [x, y, z] - center of the cell
        sphere_center: numpy array [x, y, z] - center of sphere obstacle
        sphere_radius: float - radius of the sphere
        safety_margin: float - extra clearance around obstacle

    Returns:
        True if cell is inside obstacle (should be marked as blocked), False otherwise

    Logic:
        Cell is inside if: distance(cell, sphere_center) < (radius + safety_margin)
    """

    # Calculate Euclidean distance from cell to sphere center
    distance = np.linalg.norm(cell_world_pos - sphere_center)

    # Check if within expanded obstacle radius
    threshold = sphere_radius + safety_margin
    is_inside = distance <= threshold

    return is_inside


In [ ]:
# Box obstacle collision detection

def is_cell_in_box_obstacle(cell_world_pos: np.ndarray, box_min_corner: np.ndarray, box_max_corner: np.ndarray, safety_margin: float) -> bool:
    """
    Check if a cell (at its center) is inside a box obstacle (with safety margin)

    Arguments:
        cell_world_pos: numpy array [x, y, z] - center of the cell
        box_min_corner: numpy array [x, y, z] - minimum corner of box
        box_max_corner: numpy array [x, y, z] - maximum corner of box
        safety_margin: float - extra clearance around obstacle

    Returns:
        True if cell is inside obstacle (should be marked as blocked), False otherwise

    Logic:
        Box is defined by axis-aligned bounding box (AABB)
        Cell is inside if ALL of these conditions are true:
            expanded_min.x <= cell.x <= expanded_max.x  AND
            expanded_min.y <= cell.y <= expanded_max.y  AND
            expanded_min.z <= cell.z <= expanded_max.z

        Where:
            expanded_min = box_min_corner - safety_margin (expand box outward)
            expanded_max = box_max_corner + safety_margin (expand box outward)

    Example:
        Box from [3, 3, 3] to [7, 7, 7], safety_margin = 0.5
        Expanded box: [2.5, 2.5, 2.5] to [7.5, 7.5, 7.5]

        Cell at [5, 5, 5]: inside all bounds → inside (True)
        Cell at [2.7, 5, 5]: x=2.7 in [2.5, 7.5] → inside (True)
        Cell at [2.3, 5, 5]: x=2.3 < 2.5 → outside (False)
        Cell at [8, 5, 5]: x=8 > 7.5 → outside (False)
    """

    # Expand the box by safety margin
    expanded_min = box_min_corner - safety_margin
    expanded_max = box_max_corner + safety_margin

    # Extract cell coordinates
    x, y, z = cell_world_pos[0], cell_world_pos[1], cell_world_pos[2]

    # Check if cell is within expanded box on ALL three axes
    # Cell must be >= min AND <= max for all axes to be inside
    inside_x = (expanded_min[0] <= x <= expanded_max[0])
    inside_y = (expanded_min[1] <= y <= expanded_max[1])
    inside_z = (expanded_min[2] <= z <= expanded_max[2])

    # Cell is inside box only if inside on ALL axes
    is_inside = inside_x and inside_y and inside_z

    return is_inside


### Testing the Collision Detection Functions

In [ ]:
# Test sphere collision detection
print("Testing sphere obstacle collision - ")

test_sphere_center = np.array([5.0, 5.0, 5.0])
test_sphere_radius = 1.5
test_safety = 0.5     # Taking higher value, 0.5 instead of 0.2, for testing

test_cases = [
    (np.array([5.0, 5.0, 5.0]), True, "at center"),
    (np.array([5.0, 5.0, 6.5]), True, "1.5m away (at radius boundary)"),
    (np.array([5.0, 5.0, 7.0]), True, "2.0m away (at safety boundary)"),
    (np.array([5.0, 5.0, 7.5]), False, "2.5m away (outside)"),
    (np.array([8.0, 8.0, 8.0]), False, "far away"),
]

print(f"\nSphere at {test_sphere_center}, radius={test_sphere_radius}, safety={test_safety}")
print(f"Total protected radius: {test_sphere_radius + test_safety}m\n")

all_correct = True
for pos, expected, description in test_cases:
    result = is_cell_in_sphere_obstacle(pos, test_sphere_center, test_sphere_radius, test_safety)
    status = "✓" if result == expected else "✗"
    all_correct = all_correct and (result == expected)
    print(f"  {status} Cell {description}: {result} (expected {expected})")

if all_correct:
    print("\nSphere collision detection working!")
else:
    print("\nSome test cases failed - check your logic!")


# Test box collision detection
print("\n\nTesting box obstacle collision - ")

test_box_min = np.array([3.0, 3.0, 3.0])
test_box_max = np.array([7.0, 7.0, 7.0])
test_safety = 0.5     # Taking higher value, 0.5 instead of 0.2, for testing

print(f"\nBox from {test_box_min} to {test_box_max}")
print(f"With safety margin {test_safety}, expanded to:")
print(f"  {test_box_min - test_safety} to {test_box_max + test_safety}\n")

test_cases = [
    (np.array([5.0, 5.0, 5.0]), True, "at center"),
    (np.array([3.0, 5.0, 5.0]), True, "at min boundary"),
    (np.array([7.0, 5.0, 5.0]), True, "at max boundary"),
    (np.array([2.7, 5.0, 5.0]), True, "just inside safety margin (x)"),
    (np.array([7.3, 5.0, 5.0]), True, "just inside safety margin (x)"),
    (np.array([2.3, 5.0, 5.0]), False, "outside safety margin (x)"),
    (np.array([7.7, 5.0, 5.0]), False, "outside safety margin (x)"),
    (np.array([5.0, 2.3, 5.0]), False, "outside safety margin (y)"),
    (np.array([5.0, 5.0, 2.3]), False, "outside safety margin (z)"),
    (np.array([1.0, 1.0, 1.0]), False, "far outside"),
    (np.array([9.0, 9.0, 9.0]), False, "far outside"),
    (np.array([2.7, 2.7, 2.7]), True, "corner inside safety"),
    (np.array([2.3, 2.7, 2.7]), False, "corner outside (one axis fails)"),
]

all_correct = True
for pos, expected, description in test_cases:
    result = is_cell_in_box_obstacle(pos, test_box_min, test_box_max, test_safety)
    status = "✓" if result == expected else "✗"
    all_correct = all_correct and (result == expected)
    print(f"  {status} Cell {description}: {result} (expected {expected})")

if all_correct:
    print("\nBox collision detection working!")
else:
    print("\nSome test cases failed!")
    print("Cell must be inside bounds on ALL three axes simultaneously")

### Building the Obstacle Grid for A* Algorithm
This function will mark all the cells that are in the obstacles. The algorithm will then, whn finding the route from start to goal position will know which cells are not to be traversed through.

In [ ]:
# Building the complete obstacle grid (combines spheres and boxes)

def build_obstacle_grid(obstacles: List, config: AStarConfig) -> np.ndarray:
    """
    Build a 3D boolean grid marking which cells contain obstacles

    Arguments:
        obstacles: List of obstacle objects (from helpers.get_obstacles_hard())
                  Each obstacle is either:
                  - Sphere Obstacle: has .center and .radius attributes
                  - Box Obstacle: has .min_corner and .max_corner attributes
        config: AStarConfig with grid parameters

    Returns:
        obstacle_grid: 3D numpy boolean array of shape (nx, ny, nz)
                      True = cell contains obstacle, False = free space

    Algorithm:
        1. Create 3D array of False (all cells start as free)
        2. For each cell (i, j, k) in grid:
            a. Convert to world coordinates
            b. Check against each obstacle:
               - If sphere: use is_cell_in_sphere_obstacle()
               - If box: use is_cell_in_box_obstacle()
            c. If any obstacle contains cell, mark as True
        3. Return completed grid

    """

    print(f"Building obstacle grid...")
    print(f"  Grid size: {config.nx} x {config.ny} x {config.nz} = {config.nx * config.ny * config.nz:,} cells")
    print(f"  Obstacles: {len(obstacles)}")
    print(f"  Safety margin: {config.SAFETY_MARGIN}m")

    start_time = time.time()

    # Initialize grid with all False (all free space)
    obstacle_grid = np.zeros((config.nx, config.ny, config.nz), dtype=bool)

    # Counter for blocked cells
    blocked_count = 0

    # Loop through every cell in the grid
    for i in range(config.nx):
        for j in range(config.ny):
            for k in range(config.nz):

                # Get world position of cell center
                cell_world_pos = grid_to_world(i, j, k, config)

                # Check against each obstacle
                is_blocked = False

                for obstacle in obstacles:
                    # Check if obstacle is a sphere (has 'radius' attribute)
                    if hasattr(obstacle, 'radius'):
                        # Sphere obstacle
                        sphere_center = np.array(obstacle.center)
                        sphere_radius = obstacle.radius

                        if is_cell_in_sphere_obstacle(cell_world_pos, sphere_center, sphere_radius, config.SAFETY_MARGIN):
                            is_blocked = True
                            break  # No need to check other obstacles

                    else:
                        # Box obstacle (has 'min_corner' and 'max_corner')
                        box_min = np.array(obstacle.min_corner)
                        box_max = np.array(obstacle.max_corner)

                        if is_cell_in_box_obstacle(cell_world_pos, box_min, box_max, config.SAFETY_MARGIN):
                            is_blocked = True
                            break  # No need to check other obstacles

                # Mark cell if blocked
                if is_blocked:
                    obstacle_grid[i, j, k] = True
                    blocked_count += 1
    total_cells = config.nx * config.ny * config.nz
    free_cells = total_cells - blocked_count

    print(f"\n Obstacle grid built ")
    print(f"  Blocked cells: {blocked_count:,} ({blocked_count/total_cells*100:.1f}%)")
    print(f"  Free cells: {free_cells:,} ({free_cells/total_cells*100:.1f}%)")

    return obstacle_grid


# Build the obstacle grid for hard course
obstacle_grid = build_obstacle_grid(obstacles_hard, config)


### Heuristic Function
This function calculates the estimated cost from present cell to the goal cell position. Thus our optimizer works to reduce this value along with the cell transition value.

In [ ]:
# Heuristic function for A*

def compute_heuristic(current_pos: np.ndarray, goal_pos: np.ndarray) -> float:
    """
    Compute the heuristic estimate of cost from current position to goal

    For A* to be optimal, the heuristic must be:
        h(n) <= true_cost(n, goal) for all nodes n

    Arguments:
        current_pos: numpy array [x, y, z] - current world position
        goal_pos: numpy array [x, y, z] - goal world position

    Returns:
        h_cost: float - estimated cost to reach goal

    Heuristic Choice: Euclidean Distance
        h(current, goal) = ||current - goal||
    """

    # Compute Euclidean distance (L2 norm)
    h_cost = np.linalg.norm(goal_pos - current_pos)

    return h_cost

print("\nHeuristic function working!")


### Neighbor Function
This finds the neighbours of the current cell and calculates the cost function of moving to that cell.
The cost to move is sqareroot(no. of dimensions traversed) * resolution.


In [ ]:
# Implementing neighbor generation with 26-connectivity

def get_neighbors(current_cell: GridCell, obstacle_grid: np.ndarray, config: AStarConfig) -> List[Tuple[GridCell, float]]:
    """
    Generate all valid neighbors of the current cell

    26-Connected Grid:
        In 3D, each cell can move to 26 neighbors (3³ - 1 = 26)
        Movement in each axis: -1, 0, +1

    Movement Costs:
        - Straight (1 axis): cost = 1.0 × resolution
          Examples: (0,0,1), (1,0,0), (0,-1,0)

        - Diagonal 2D (2 axes): cost = √2 × resolution
          Examples: (1,1,0), (1,0,1), (0,1,-1)

        - Diagonal 3D (3 axes): cost = √3 × resolution
          Examples: (1,1,1), (1,-1,1), (-1,1,-1)

    Arguments:
        current_cell: GridCell object (current position in grid)
        obstacle_grid: 3D boolean array (True = obstacle)
        config: AStarConfig with grid parameters

    Returns:
        neighbors: List of tuples (neighbor_cell, movement_cost)
                  Only includes valid, collision-free neighbors

    Algorithm:
        1. Generate all 26 possible movements (dx, dy, dz) where each ∈ {-1, 0, 1}
        2. Skip (0, 0, 0) - that's the current cell
        3. For each movement:
            a. Calculate neighbor indices: (i+dx, j+dy, k+dz)
            b. Check if valid (within grid bounds)
            c. Check if free (not an obstacle)
            d. Calculate movement cost based on number of non-zero axes
            e. Create neighbor GridCell and add to list
        4. Return list of (neighbor, cost) tuples
    """

    neighbors = []

    # Try all 26 possible movements
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:

                # Skip the current cell (0, 0, 0)
                if dx == 0 and dy == 0 and dz == 0:
                    continue

                # Calculate neighbor grid coordinates
                ni = current_cell.i + dx
                nj = current_cell.j + dy
                nk = current_cell.k + dz

                # Check if neighbor is within grid bounds
                if not is_valid_cell(ni, nj, nk, config):
                    continue

                # Check if neighbor is blocked by obstacle
                if obstacle_grid[ni, nj, nk]:
                    continue

                # Calculate movement cost based on distance
                # Count how many axes we're moving along
                num_axes = (abs(dx) + abs(dy) + abs(dz))

                if num_axes == 1:
                    # Straight movement (one axis)
                    movement_cost = config.COST_STRAIGHT * config.GRID_RESOLUTION
                elif num_axes == 2:
                    # Diagonal 2D (two axes)
                    movement_cost = config.COST_DIAGONAL_2D * config.GRID_RESOLUTION
                else:  # num_axes == 3
                    # Diagonal 3D (three axes)
                    movement_cost = config.COST_DIAGONAL_3D * config.GRID_RESOLUTION

                # Create neighbor cell
                neighbor_world_pos = grid_to_world(ni, nj, nk, config)
                neighbor_cell = GridCell(ni, nj, nk,
                                        neighbor_world_pos[0],
                                        neighbor_world_pos[1],
                                        neighbor_world_pos[2])

                # Add to neighbors list
                neighbors.append((neighbor_cell, movement_cost))

    return neighbors

print("\nNeighbor generation working!")

### Main Function - Implementing the A* Algorithm
Below is the main fucntion that defines the core of the A* search algorithm.

In [ ]:
# The Main A* Search Algorithm

def a_star_search(start_pos: np.ndarray, goal_pos: np.ndarray, obstacle_grid: np.ndarray, config: AStarConfig) -> Tuple[Optional[List[np.ndarray]], Dict]:
    """
    A* path planning algorithm in 3D grid

    Arguments:
        start_pos: numpy array [x, y, z] - start position in world coordinates
        goal_pos: numpy array [x, y, z] - goal position in world coordinates
        obstacle_grid: 3D boolean array - True = obstacle
        config: AStarConfig object

    Returns:
        path: List of world positions [x, y, z] from start to goal (or None if no path)
        info: Dictionary with statistics:
            - 'success': bool
            - 'nodes_expanded': int
            - 'path_length': float
            - 'planning_time': float

    A* Algorithm:
        1. Initialize open list (priority queue) with start node
        2. Initialize closed set (visited nodes)
        3. While open list not empty:
            a. Pop node with lowest f_cost from open list
            b. If node is goal, reconstruct path and return
            c. Add node to closed set
            d. For each neighbor:
                - Skip if in closed set
                - Calculate tentative g_cost
                - If neighbor not in open list OR found better path:
                    * Update neighbor costs and parent
                    * Add to open list
        4. If open list empty, no path exists
        We use (f_cost, counter, cell) to break ties consistently
    """

    print(f"\n{'='*60}")
    print(f" A* SEARCH")
    print(f"{'='*60}")
    print(f"Start: {start_pos}")
    print(f"Goal:  {goal_pos}")

    start_time = time.time()

    # Convert world positions to grid coordinates
    start_grid = world_to_grid(start_pos, config)
    goal_grid = world_to_grid(goal_pos, config)

    # Create start and goal cells
    start_world = grid_to_world(start_grid[0], start_grid[1], start_grid[2], config)
    goal_world = grid_to_world(goal_grid[0], goal_grid[1], goal_grid[2], config)

    start_cell = GridCell(start_grid[0], start_grid[1], start_grid[2],
                         start_world[0], start_world[1], start_world[2])
    goal_cell = GridCell(goal_grid[0], goal_grid[1], goal_grid[2],
                        goal_world[0], goal_world[1], goal_world[2])

    # Check if start or goal is blocked
    if obstacle_grid[start_grid[0], start_grid[1], start_grid[2]]:
        print("Start position is blocked by obstacle!")
        return None, {'success': False, 'nodes_expanded': 0, 'path_length': 0, 'planning_time': 0}

    if obstacle_grid[goal_grid[0], goal_grid[1], goal_grid[2]]:
        print("Goal position is blocked by obstacle!")
        return None, {'success': False, 'nodes_expanded': 0, 'path_length': 0, 'planning_time': 0}

    # Initialize start cell costs
    start_cell.g_cost = 0.0
    start_cell.h_cost = compute_heuristic(start_cell.world_position(), goal_cell.world_position())
    start_cell.f_cost = start_cell.g_cost + start_cell.h_cost
    start_cell.parent = None

    # Priority queue (open list): stores (f_cost, counter, cell)
    # Counter ensures consistent ordering when f_costs are equal
    open_list = []
    counter = 0  # Tie-breaker for equal f_costs
    heapq.heappush(open_list, (start_cell.f_cost, counter, start_cell))
    counter += 1

    # Closed set: stores grid positions (i, j, k) of visited nodes
    closed_set = set()

    # Open set: for quick lookup of cells in open list
    # Maps (i, j, k) -> cell
    open_set = {start_cell.position_tuple(): start_cell}

    # Statistics
    nodes_expanded = 0

    # Main A* loop
    while open_list:

        # Pop node with lowest f_cost
        current_f, _, current_cell = heapq.heappop(open_list)

        # Remove from open set
        current_pos_tuple = current_cell.position_tuple()
        if current_pos_tuple in open_set:
            del open_set[current_pos_tuple]

        # Check if reached goal
        if current_cell == goal_cell:
            # Success! Reconstruct path
            path = []
            node = current_cell
            while node is not None:
                path.append(node.world_position())
                node = node.parent
            path.reverse()  # Reverse to get start -> goal order

            planning_time = time.time() - start_time
            path_length = sum(np.linalg.norm(path[i+1] - path[i]) for i in range(len(path)-1))

            print(f"\n Path found!")
            print(f"   Nodes expanded: {nodes_expanded}")
            print(f"   Path length: {path_length:.2f}m")
            print(f"   Planning time: {planning_time:.3f}s")
            print(f"{'='*60}")

            return path, {
                'success': True,
                'nodes_expanded': nodes_expanded,
                'path_length': path_length,
                'planning_time': planning_time
            }

        # Add to closed set
        closed_set.add(current_pos_tuple)
        nodes_expanded += 1

        # Get neighbors
        neighbors = get_neighbors(current_cell, obstacle_grid, config)

        # Process each neighbor
        for neighbor_cell, move_cost in neighbors:

            neighbor_pos_tuple = neighbor_cell.position_tuple()

            # Skip if already visited
            if neighbor_pos_tuple in closed_set:
                continue

            # Calculate tentative g_cost
            tentative_g = current_cell.g_cost + move_cost

            # Check if neighbor is in open set
            if neighbor_pos_tuple in open_set:
                # Neighbor already in open list
                existing_neighbor = open_set[neighbor_pos_tuple]

                # Check if we found a better path
                if tentative_g < existing_neighbor.g_cost:
                    # Update with better path
                    existing_neighbor.g_cost = tentative_g
                    existing_neighbor.f_cost = existing_neighbor.g_cost + existing_neighbor.h_cost
                    existing_neighbor.parent = current_cell

                    # Re-add to priority queue with new f_cost
                    heapq.heappush(open_list, (existing_neighbor.f_cost, counter, existing_neighbor))
                    counter += 1

            else:
                # New neighbor - add to open list
                neighbor_cell.g_cost = tentative_g
                neighbor_cell.h_cost = compute_heuristic(neighbor_cell.world_position(), goal_cell.world_position())
                neighbor_cell.f_cost = neighbor_cell.g_cost + neighbor_cell.h_cost
                neighbor_cell.parent = current_cell

                # Add to open list and open set
                heapq.heappush(open_list, (neighbor_cell.f_cost, counter, neighbor_cell))
                counter += 1
                open_set[neighbor_pos_tuple] = neighbor_cell

    # If we get here, open list is empty - no path exists
    planning_time = time.time() - start_time

    print(f"\n No path found!")
    print(f"   Nodes expanded: {nodes_expanded}")
    print(f"   Planning time: {planning_time:.3f}s")
    print(f"{'='*60}")


    return None, {
        'success': False,
        'nodes_expanded': nodes_expanded,
        'path_length': 0,
        'planning_time': planning_time
    }


print(" A* algorithm implemented!")

### Converting the A* Path to the Pose3 Path Configuration
This function changes only Yaw values according to the present and previous grid cell positions. Only if the goal position is provided, then only it changes the final orientation to the goal orientation else it always keeps the previous orientation when reaching the new grid point. This also applies in our current case when it has to just traverse through the 4 hoops and not specifically reach a target pose and position.


In [ ]:
def grid_path_to_pose3_path(grid_path: List[np.ndarray], goal_pose: gtsam.Pose3 = None) -> List[gtsam.Pose3]:
    """
    Convert A* grid path to Pose3 trajectory

    Arguments:
        grid_path: List of [x, y, z] positions
        goal_pose: Optional target pose (if provided, uses its orientation at end)
    """
    pose_path = []

    for i in range(len(grid_path)):
        position = gtsam.Point3(grid_path[i][0], grid_path[i][1], grid_path[i][2])

        if i < len(grid_path) - 1:
            # Not last: direction-based orientation
            direction = grid_path[i+1] - grid_path[i]
            yaw = np.arctan2(direction[1], direction[0])
            pitch = 0.0
            roll = 0.0
            attitude = gtsam.Rot3.Ypr(yaw, pitch, roll)

        elif goal_pose is not None:
            # Last waypoint AND goal pose provided: use goal orientation
            attitude = goal_pose.rotation()

        else:
            # Last waypoint, no goal pose: use previous orientation
            if len(pose_path) > 0:
                attitude = pose_path[-1].rotation()
            else:
                attitude = gtsam.Rot3()  # Identity

        pose = gtsam.Pose3(attitude, position)
        pose_path.append(pose)

    return pose_path


## Implementing the A* Algorithm

In [ ]:
# Defining the Function

def drone_racing_astar_with_obstacles(start: gtsam.Pose3, targets: List[gtsam.Pose3],obstacles: List,config: AStarConfig, obstacle_grid: np.ndarray) -> Tuple[List[gtsam.Pose3], Dict]:
    """
    Plan complete drone racing path using A* through multiple hoops

    Arguments:
        start: gtsam.Pose3 - starting pose
        targets: List[gtsam.Pose3] - hoop center targets (in order)
        obstacles: List of obstacle objects
        config: AStarConfig
        obstacle_grid: 3D boolean array of obstacles

    Returns:
        complete_path: List[gtsam.Pose3] - full path through all hoops
        stats: Dict with planning statistics

    Algorithm:
        1. For each hoop target:
            a. Run A* from current position to hoop center
            b. Convert grid path to Pose3 path
            c. Add hoop passage waypoints (helpers.pass_through_the_hoop)
            d. Append to complete path
            e. Update current position
        2. Track statistics (total time, segments, etc.)

    """

    complete_path = []
    current_position = start

    # Statistics
    total_planning_time = 0
    total_nodes_expanded = 0
    segment_count = 0
    all_segments_successful = True

    # Plan path segment for each hoop
    for idx, target in enumerate(targets):
        print(f"\n{'─'*60}")
        print(f"Planning segment {idx+1}/{len(targets)} to hoop at {target.translation()}")
        print(f"{'─'*60}")

        segment_start_time = time.time()

        # Run A* from current position to target
        grid_path, info = a_star_search(
            start_pos=np.array([current_position.translation()[0],
                               current_position.translation()[1],
                               current_position.translation()[2]]),
            goal_pos=np.array([target.translation()[0],
                              target.translation()[1],
                              target.translation()[2]]),
            obstacle_grid=obstacle_grid,
            config=config
        )

        segment_time = time.time() - segment_start_time

        # Check if segment planning succeeded
        if grid_path is None or not info['success']:
            print(f"Failed to find path to hoop {idx+1}")
            all_segments_successful = False
            break

        # Convert to Pose3 path
        segment_pose_path = grid_path_to_pose3_path(grid_path)

        # Add hoop passage waypoints
        helpers.pass_through_the_hoop(target, segment_pose_path)

        # Append to complete path (avoid duplicating first point)
        if len(complete_path) == 0:
            complete_path.extend(segment_pose_path)
        else:
            # Skip first point of segment (same as last point of previous segment)
            complete_path.extend(segment_pose_path[1:])

        # Update current position for next segment
        current_position = complete_path[-1]

        # Update statistics
        total_planning_time += segment_time
        total_nodes_expanded += info['nodes_expanded']
        segment_count += 1

        print(f" Segment {idx+1} complete:")
        print(f"   Waypoints: {len(segment_pose_path)}")
        print(f"   Path length: {info['path_length']:.2f}m")
        print(f"   Planning time: {segment_time:.3f}s")

    # Final statistics
    stats = {
        'success': all_segments_successful,
        'total_planning_time': total_planning_time,
        'total_nodes_expanded': total_nodes_expanded,
        'segments_completed': segment_count,
        'total_waypoints': len(complete_path)
    }

    print("\n" + "=" * 80)
    if all_segments_successful:
        print("✅ COMPLETE PATH PLANNED SUCCESSFULLY!")
        print(f"   Total waypoints: {len(complete_path)}")
        print(f"   Total planning time: {total_planning_time:.3f}s")
        print(f"   Total nodes expanded: {total_nodes_expanded:,}")
        print(f"   Segments: {segment_count}/{len(targets)}")
    else:
        print("❌ PATH PLANNING FAILED")
        print(f"   Completed segments: {segment_count}/{len(targets)}")
    print("=" * 80)

    return complete_path if all_segments_successful else None, stats


In [ ]:
# Executing A* on the complete 8-obstacle hard course!
print("\n" + "=" * 80)
print("🏁 FINAL CHALLENGE: A* ON 8-OBSTACLE HARD COURSE 🏁\n")
print("🏁 DRONE RACING WITH A* - PLANNING PATH")
print("=" * 80)

# Run A* drone racing
astar_start_time = time.time()

astar_racing_path, astar_stats = drone_racing_astar_with_obstacles(
    start=start_pose,
    targets=targets,
    obstacles=obstacles_hard,
    config=config,
    obstacle_grid=obstacle_grid
)

astar_total_time = time.time() - astar_start_time

# Display results
if astar_racing_path:
    print("\n" + "🎉" * 40)
    print("SUCCESS! A* COMPLETED THE HARD COURSE!")
    print("🎉" * 40)

    # Validate collision-free
    print("\nValidating path...")
    has_collision, collision_details = helpers.check_path_collision( astar_racing_path, obstacles_hard)

    if not has_collision:
        print("✅ Path is COLLISION-FREE!")
        print("✅ All hoops reached!")
    else:
        print(f"⚠ Path has collisions: {collision_details}")

    # Path statistics
    print("\n📊 PATH STATISTICS:")
    print(f"   Total waypoints: {len(astar_racing_path)}")
    print(f"   Planning time: {astar_total_time:.3f}s")
    print(f"   Nodes expanded: {astar_stats['total_nodes_expanded']:,}")

    # Calculate path length
    path_length = 0
    for i in range(len(astar_racing_path) - 1):
        p1 = astar_racing_path[i].translation()
        p2 = astar_racing_path[i+1].translation()
        path_length += np.linalg.norm(np.array([p2[0]-p1[0], p2[1]-p1[1], p2[2]-p1[2]]))

    print(f"   Path length: {path_length:.2f}m")

else:
    print("\n❌ A* failed to complete the course")
    print(f"   Segments completed: {astar_stats['segments_completed']}/{len(targets)}")

## Visualizing the Results

In [ ]:
# Visualize the complete A* racing path with hoops and obstacles

if astar_racing_path:
    print("\nVisualizing A* racing path...\n")

    fig = helpers.drone_racing_path_with_obstacles(
        hoops=hoops,
        start=start_pose,
        path=astar_racing_path,
        obstacles=obstacles_hard,
    )

    fig

    print("✅ Visualization complete!")
    print("\n🎮 Interactive controls:")
    print("   - Rotate: Click and drag")
    print("   - Zoom: Scroll wheel")
    print("   - Pan: Right-click and drag")
else:
    print("⚠ No path to visualize")

### Smoothing

In [ ]:
## Path Smoothing: Remove redundant waypoints while preserving hoop passages

def smooth_path_shortcut_with_hoops(path: List[gtsam.Pose3],
                                    obstacles: List,
                                    hoops: List[gtsam.Pose3],
                                    max_iterations: int = 100) -> List[gtsam.Pose3]:
    """
    Smooth path by removing unnecessary intermediate waypoints
    WHILE PRESERVING HOOP PASSAGE WAYPOINTS

    Algorithm (Hoop-Aware Shortcutting):
        1. Identify which waypoints are near hoops (must be kept)
        2. Try to connect non-adjacent waypoints directly
        3. If direct connection is collision-free AND doesn't skip hoops, remove intermediate
        4. Repeat until no more shortcuts possible

    This reduces waypoint count without breaking hoop passages

    Arguments:
        path: List[gtsam.Pose3] - original jagged path
        obstacles: List of obstacles to check collisions
        hoops: List[gtsam.Pose3] - hoop positions to preserve
        max_iterations: maximum shortcutting attempts

    Returns:
        smoothed_path: List[gtsam.Pose3] - smoother path with fewer waypoints
    """

    if len(path) <= 2:
        return path

    print(f"\n{'='*60}")
    print(f"🔧 PATH SMOOTHING (HOOP-AWARE)")
    print(f"{'='*60}")
    print(f"Original path: {len(path)} waypoints")
    print(f"Hoops to preserve: {len(hoops)}")

    # Step 1: Identify hoop waypoints (must NOT be removed)
    hoop_waypoint_indices = set()
    hoop_threshold = 0.3  # Within 0.3m of hoop = hoop waypoint

    for hoop_idx, hoop in enumerate(hoops):
        hoop_pos = np.array([hoop.translation()[0],
                            hoop.translation()[1],
                            hoop.translation()[2]])

        # Find closest waypoint to this hoop
        min_dist = float('inf')
        closest_idx = -1

        for wp_idx, waypoint in enumerate(path):
            wp_pos = np.array([waypoint.translation()[0],
                              waypoint.translation()[1],
                              waypoint.translation()[2]])
            dist = np.linalg.norm(wp_pos - hoop_pos)

            if dist < min_dist:
                min_dist = dist
                closest_idx = wp_idx

        # Mark this waypoint as protected if it's close to hoop
        if min_dist < hoop_threshold:
            hoop_waypoint_indices.add(closest_idx)
            print(f"   Protected waypoint {closest_idx} (hoop {hoop_idx+1}, dist={min_dist:.2f}m)")

    print(f"   Total protected waypoints: {len(hoop_waypoint_indices)}")

    # Step 2: Shortcut smoothing (avoiding protected waypoints)
    smoothed = list(path)  # Copy path

    # Adjust protected indices as we remove waypoints
    protected_indices = sorted(hoop_waypoint_indices)

    iteration = 0
    improved = True

    while improved and iteration < max_iterations:
        improved = False
        i = 0

        while i < len(smoothed) - 2:
            # Check if we're trying to remove a protected waypoint
            if (i + 1) in protected_indices:
                # Can't remove this waypoint - it's a hoop passage point
                i += 1
                continue

            # Try to shortcut from waypoint i to i+2 (skip i+1)
            pos_i = np.array([smoothed[i].translation()[0],
                             smoothed[i].translation()[1],
                             smoothed[i].translation()[2]])
            pos_i2 = np.array([smoothed[i+2].translation()[0],
                              smoothed[i+2].translation()[1],
                              smoothed[i+2].translation()[2]])

            # Check collision on direct segment
            has_collision = helpers.check_segment_collision(pos_i, pos_i2, obstacles)

            if not has_collision:
                # Can shortcut! Remove intermediate waypoint
                smoothed.pop(i+1)
                improved = True

                # Update protected indices (shift down by 1)
                protected_indices = [idx if idx <= i else idx - 1 for idx in protected_indices]

                # Don't increment i, check again from same position
            else:
                i += 1

        iteration += 1

    print(f"\nSmoothed path: {len(smoothed)} waypoints")
    print(f"Removed: {len(path) - len(smoothed)} waypoints ({(len(path) - len(smoothed))/len(path)*100:.1f}%)")
    print(f"Protected waypoints preserved: {len(protected_indices)}")
    print(f"Iterations: {iteration}")
    print(f"{'='*60}")

    return smoothed


# Apply hoop-aware smoothing to A* path
if astar_racing_path:
    print("\nApplying hoop-aware path smoothing...")

    astar_path_smoothed = smooth_path_shortcut_with_hoops(
        path=astar_racing_path,
        obstacles=obstacles_hard,
        hoops=hoops,  # Pass hoops!
        max_iterations=50
    )

    # Calculate path lengths
    def compute_path_length(path):
        length = 0
        for i in range(len(path) - 1):
            p1 = path[i].translation()
            p2 = path[i+1].translation()
            length += np.linalg.norm(np.array([p2[0]-p1[0], p2[1]-p1[1], p2[2]-p1[2]]))
        return length

    original_length = compute_path_length(astar_racing_path)
    smoothed_length = compute_path_length(astar_path_smoothed)

    print(f"\n📊 Smoothing Results:")
    print(f"   Original: {len(astar_racing_path)} waypoints, {original_length:.2f}m")
    print(f"   Smoothed: {len(astar_path_smoothed)} waypoints, {smoothed_length:.2f}m")
    print(f"   Path shortened by: {original_length - smoothed_length:.2f}m ({(original_length - smoothed_length)/original_length*100:.1f}%)")

    # Verify still collision-free
    has_collision_smooth, _ = helpers.check_path_collision(astar_path_smoothed, obstacles_hard)
    print(f"   Collision-free: {' Yes' if not has_collision_smooth else ' No'}")

    # IMPORTANT: Verify all hoops are still reached
    print(f"\n Hoop Verification:")
    hoops_reached = 0
    for hoop_idx, hoop in enumerate(hoops):
        hoop_pos = np.array([hoop.translation()[0],
                            hoop.translation()[1],
                            hoop.translation()[2]])

        # Check if any waypoint is close to this hoop
        min_dist = float('inf')
        for waypoint in astar_path_smoothed:
            wp_pos = np.array([waypoint.translation()[0],
                              waypoint.translation()[1],
                              waypoint.translation()[2]])
            dist = np.linalg.norm(wp_pos - hoop_pos)
            min_dist = min(min_dist, dist)

        if min_dist < 0.5:  # Within 0.5m = passed through
            hoops_reached += 1
            print(f"  Hoop {hoop_idx+1}: closest waypoint {min_dist:.2f}m")
        else:
            print(f"  Hoop {hoop_idx+1}: closest waypoint {min_dist:.2f}m (TOO FAR!)")

    print(f"\n   Total hoops reached: {hoops_reached}/{len(hoops)}")

    if hoops_reached == len(hoops):
        print("\nPath smoothing complete - ALL HOOPS PRESERVED!")
    else:
        print(" Consider using original path or adjusting protection threshold")

else:
    print(" Yayay! No path to smooth")

In [ ]:
# Overlaying both paths in one plot

if astar_racing_path and astar_path_smoothed:
    print("\nCreating overlay comparison...\n")

    # First, show smoothed path (it will be the base)
    helpers.drone_racing_path_with_obstacles(
        hoops=hoops,
        start=start_pose,
        path=astar_path_smoothed,
        obstacles=obstacles_hard,
       )

    # Note: The helper function doesn't return the figure, so we can't overlay
    # But showing them sequentially is actually clearer for comparison!

    print("\n" + "-"*80)

    # Then show original path for comparison
    helpers.drone_racing_path_with_obstacles(
        hoops=hoops,
        start=start_pose,
        path=astar_racing_path,
        obstacles=obstacles_hard,
        )

    print("\n✅ Both visualizations shown - compare the waypoint density!")

else:
    print("⚠ No path available")